# HyWeCS 

Every convolutional output filter contains exactly `n` trainable base weights: `r` are stored in the ORaW partition and the complementary `n-r` are stored in the SANW partition. The two partitions are assembled only transiently for convolution; no duplicated full ORaW/SHANW kernel parameters are created.


In [ ]:

import math
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense

tf.keras.backend.clear_session()
tf.random.set_seed(0)
np.random.seed(0)


In [ ]:
import tensorflow as tf
#tf.keras.mixed_precision.set_global_policy('mixed_float16')

import tensorflow as tf
import numpy as np
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, ReLU, Add
from tensorflow.keras.models import Model
from tensorflow.keras.losses import KLDivergence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import DenseNet121, ResNet50V2
from tensorflow.keras.layers import GlobalAveragePooling2D
import copy
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, ReLU, Add
from tensorflow.keras.models import Model
from tensorflow.keras.losses import KLDivergence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import DenseNet169, MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D
import copy

import tensorflow as tf
from tensorflow.keras import layers
import tensorflow as tf
from tensorflow.keras import layers

### Input Multimodal data

In [ ]:
X_train_image = np.load('x_train.npy')
X_train_ehr = np.load('x_train.npy')
y_train = np.load('y_train.npy')

X_val_image = np.load('x_val.npy')
X_val_ehr = np.load('x_val.npy')
y_val = np.load('y_val.npy')

X_test_image = np.load('x_test(1).npy')
X_test_ehr = np.load('x_test(1).npy')
y_test = np.load('y_test(1).npy')

X_train_image.shape, X_train_ehr.shape, y_train.shape, X_val_image.shape, X_val_ehr.shape, y_val.shape, X_test_image.shape, X_test_ehr.shape, y_test.shape

## 1. Paper-aligned HyWeCS implementation

The following implementation changes the defense/model only; it does not replace, relocate, reshape, or auto-discover the dataset loaded above.


In [ ]:

from __future__ import annotations

from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any, Iterable, Mapping, Optional, Sequence

import math
import numpy as np
import tensorflow as tf
from scipy.stats import beta as scipy_beta
from scipy.stats import norm as scipy_norm

keras = tf.keras
layers = tf.keras.layers
Tensor = tf.Tensor

ABSTAIN = -1


# =============================================================================
# Configuration and validation
# =============================================================================


@dataclass(frozen=True)
class HyWeCSLayerConfig:
    
    filters: int
    kernel_size: int | tuple[int, int] = 3
    strides: int | tuple[int, int] = 1
    padding: str = "same"
    use_bias: bool = False

    # Within-filter allocation.
    oraw_fraction: float = 0.5
    allocation_seed: int = 0

    # ORaW / LOT.
    sigma_rp: float = 0.05
    lot_newton_steps: int = 4
    lot_eps: float = 1e-4

    # SHANW.
    shanw_recursions: int = 3
    curvature: float = 1.0
    hyperbolic_dim: int = 5
    channel_bottleneck_ratio: int = 8
    pi_init: float = 0.05
    rho_logit_init: float = -2.0

    # Shared layer-wise calibration.
    tau: float = 1.0
    calibration_mode: str = "diagnostic"  # {"diagnostic", "verified"}
    feature_norm_bound: Optional[float] = None
    attention_lipschitz_bound: Optional[float] = None

    # Stateless stochastic realization.
    seed_salt: int = 0

    def normalized_kernel_size(self) -> tuple[int, int]:
        if isinstance(self.kernel_size, int):
            return (self.kernel_size, self.kernel_size)
        return tuple(int(v) for v in self.kernel_size)

    def normalized_strides(self) -> tuple[int, int]:
        if isinstance(self.strides, int):
            return (self.strides, self.strides)
        return tuple(int(v) for v in self.strides)

    def validate(self) -> None:
        kh, kw = self.normalized_kernel_size()
        sh, sw = self.normalized_strides()
        if self.filters <= 0 or kh <= 0 or kw <= 0 or sh <= 0 or sw <= 0:
            raise ValueError("filters, kernel_size, and strides must be positive.")
        if not 0.0 < self.oraw_fraction < 1.0:
            raise ValueError("oraw_fraction must lie strictly between 0 and 1.")
        if self.sigma_rp < 0.0:
            raise ValueError("sigma_rp must be nonnegative.")
        if self.lot_newton_steps <= 0 or self.lot_eps <= 0.0:
            raise ValueError("LOT Newton steps and epsilon must be positive.")
        if self.shanw_recursions <= 0:
            raise ValueError("shanw_recursions must be positive.")
        if self.curvature <= 0.0 or self.hyperbolic_dim <= 0:
            raise ValueError("curvature and hyperbolic_dim must be positive.")
        if self.channel_bottleneck_ratio <= 0:
            raise ValueError("channel_bottleneck_ratio must be positive.")
        if not 0.0 < self.tau <= 1.0:
            raise ValueError("tau must lie in (0, 1].")
        if self.calibration_mode not in {"diagnostic", "verified"}:
            raise ValueError("calibration_mode must be 'diagnostic' or 'verified'.")
        if self.calibration_mode == "verified":
            if self.feature_norm_bound is None or self.feature_norm_bound <= 0.0:
                raise ValueError(
                    "verified mode requires a sound positive feature_norm_bound."
                )
            if (
                self.attention_lipschitz_bound is None
                or self.attention_lipschitz_bound <= 0.0
            ):
                raise ValueError(
                    "verified mode requires a sound positive "
                    "attention_lipschitz_bound."
                )


@dataclass(frozen=True)
class HyWeCSModelConfig:
    

    input_shape_modality_1: tuple[int, int, int]
    input_shape_modality_2: tuple[int, int, int]
    num_classes: int

    base_width: int = 64
    blocks_per_stage: tuple[int, int, int, int] = (2, 2, 2, 2)
    stage_width_multipliers: tuple[int, int, int, int] = (1, 2, 4, 8)

    oraw_fraction: float = 0.5
    sigma_rp: float = 0.05
    shanw_recursions: int = 3
    curvature: float = 1.0
    hyperbolic_dim: int = 5
    channel_bottleneck_ratio: int = 8
    tau: float = 1.0
    lot_newton_steps: int = 4
    lot_eps: float = 1e-4

    calibration_mode: str = "diagnostic"
    feature_norm_bound: Optional[float] = None
    attention_lipschitz_bound: Optional[float] = None

    # The paper uses L_star=2. A sound bound on the uncalibrated complete
    # learner is still required to compute the global scaling factor.
    global_lipschitz_target: float = 2.0
    verified_network_bound: Optional[float] = None
    reciprocal_cross_fusion_verified: bool = False

    use_hywecs_stem: bool = False
    allocation_seed: int = 0
    seed: int = 0

    def validate(self) -> None:
        if self.num_classes <= 1:
            raise ValueError("num_classes must be at least 2.")
        if self.base_width <= 0:
            raise ValueError("base_width must be positive.")
        if len(self.blocks_per_stage) != 4 or len(self.stage_width_multipliers) != 4:
            raise ValueError("The reference ResNet-18 configuration has four stages.")
        if any(v <= 0 for v in self.blocks_per_stage):
            raise ValueError("Each stage must contain at least one residual block.")
        if self.global_lipschitz_target != 2.0:
            raise ValueError(
                "The current manuscript fixes the certified global target to L_star=2."
            )
        # Reuse layer validation for shared settings.
        HyWeCSLayerConfig(
            filters=self.base_width,
            oraw_fraction=self.oraw_fraction,
            sigma_rp=self.sigma_rp,
            shanw_recursions=self.shanw_recursions,
            curvature=self.curvature,
            hyperbolic_dim=self.hyperbolic_dim,
            channel_bottleneck_ratio=self.channel_bottleneck_ratio,
            tau=self.tau,
            lot_newton_steps=self.lot_newton_steps,
            lot_eps=self.lot_eps,
            calibration_mode=self.calibration_mode,
            feature_norm_bound=self.feature_norm_bound,
            attention_lipschitz_bound=self.attention_lipschitz_bound,
        ).validate()
        if self.calibration_mode == "verified":
            if self.verified_network_bound is None or self.verified_network_bound <= 0:
                raise ValueError(
                    "verified mode requires a sound positive verified_network_bound."
                )
            if not self.reciprocal_cross_fusion_verified:
                raise ValueError(
                    "verified mode requires the paper's reciprocal cross-fusion "
                    "condition to be independently verified."
                )


@dataclass(frozen=True)
class CertificationResult:
    expected_logit_class: int
    expected_logit_radius_mc: float
    smoothed_class: int
    randomized_smoothing_radius: float
    p_lower: float
    pilot_counts: np.ndarray
    certification_counts: np.ndarray
    expected_scores: np.ndarray
    abstained: bool


# =============================================================================
# Seed utilities
# =============================================================================


def _as_int32_seed(seed: Tensor | Sequence[int]) -> Tensor:
    seed_tensor = tf.convert_to_tensor(seed, dtype=tf.int32)
    if seed_tensor.shape.rank != 1 or seed_tensor.shape[-1] != 2:
        raise ValueError("A scalar stateless seed must have shape [2].")
    return seed_tensor


def canonicalize_seed_batch(
    seed: Optional[Tensor | Sequence[int]], batch_size: Tensor
) -> Tensor:
    """Return independent stateless seeds with shape [B, 2]."""

    if seed is None:
        return tf.random.uniform(
            shape=tf.stack([batch_size, 2]),
            minval=0,
            maxval=np.iinfo(np.int32).max,
            dtype=tf.int32,
        )

    seed_tensor = tf.convert_to_tensor(seed, dtype=tf.int32)
    if seed_tensor.shape.rank == 1:
        if seed_tensor.shape[-1] != 2:
            raise ValueError("seed must have shape [2] or [B, 2].")
        return tf.random.experimental.stateless_split(seed_tensor, num=batch_size)
    if seed_tensor.shape.rank == 2 and seed_tensor.shape[-1] == 2:
        tf.debugging.assert_equal(
            tf.shape(seed_tensor)[0], batch_size, message="seed batch mismatch"
        )
        return seed_tensor
    raise ValueError("seed must have shape [2] or [B, 2].")


def fold_in_seed_batch(seeds: Tensor, salt: int) -> Tensor:
    """Fold a deterministic integer into every seed in [B,2]."""

    return tf.map_fn(
        lambda s: tf.random.experimental.stateless_fold_in(s, int(salt)),
        seeds,
        fn_output_signature=tf.TensorSpec(shape=(2,), dtype=tf.int32),
    )


def stateless_normal_per_example(
    seeds: Tensor,
    sample_shape: tuple[int, ...],
    dtype: tf.dtypes.DType,
) -> Tensor:
    """Draw one independent tensor of ``sample_shape`` for every batch seed."""

    return tf.map_fn(
        lambda s: tf.random.stateless_normal(sample_shape, seed=s, dtype=dtype),
        seeds,
        fn_output_signature=tf.TensorSpec(shape=sample_shape, dtype=dtype),
    )


def random_seed_batch(batch_size: int | Tensor) -> Tensor:
    return tf.random.uniform(
        shape=tf.stack([tf.cast(batch_size, tf.int32), 2]),
        minval=0,
        maxval=np.iinfo(np.int32).max,
        dtype=tf.int32,
    )


# =============================================================================
# Basic tensor and norm helpers
# =============================================================================


def make_fixed_within_filter_indices(
    kernel_size: tuple[int, int],
    in_channels: int,
    out_channels: int,
    oraw_fraction: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Return the two disjoint location partitions of every output filter.

    For each output filter, the flattened convolution kernel has
    ``n = kh * kw * Cin`` positions. Exactly ``r`` locations are assigned to
    ORaW and the complementary ``n-r`` locations are assigned to SHANW. The
    returned arrays have shapes ``[Cout, r]`` and ``[Cout, n-r]``. They are
    deterministic metadata only; they are not trainable tensors and do not
    introduce extra kernel parameters.
    """

    kh, kw = kernel_size
    n = int(kh * kw * in_channels)
    r = int(np.rint(float(oraw_fraction) * n))
    r = min(max(r, 1), n - 1)

    oraw = np.empty((out_channels, r), dtype=np.int32)
    shanw = np.empty((out_channels, n - r), dtype=np.int32)
    all_locations = np.arange(n, dtype=np.int32)

    for out_index in range(out_channels):
        rng = np.random.default_rng(int(seed) + 104729 * out_index)
        selected = np.sort(rng.choice(n, size=r, replace=False).astype(np.int32))
        complement = np.setdiff1d(all_locations, selected, assume_unique=True)
        oraw[out_index] = selected
        shanw[out_index] = complement

    return oraw, shanw

def same_padding_conv_output_shape(
    input_shape: tf.TensorShape | Sequence[Optional[int]],
    out_channels: int,
    strides: tuple[int, int] | int = (1, 1),
) -> tf.TensorShape:
    

    shape = tf.TensorShape(input_shape)
    if shape.rank != 4:
        raise ValueError(f"Expected NHWC rank-4 shape, got {shape}.")
    if isinstance(strides, int):
        sh = sw = int(strides)
    else:
        sh, sw = int(strides[0]), int(strides[1])

    def out_dim(dim: Optional[int], stride: int) -> Optional[int]:
        if dim is None:
            return None
        return int(math.ceil(int(dim) / int(stride)))

    return tf.TensorShape(
        [shape[0], out_dim(shape[1], sh), out_dim(shape[2], sw), int(out_channels)]
    )


def convolution_operator_upper_bound(kernel: Tensor) -> Tensor:
    
    kernel = tf.convert_to_tensor(kernel)
    if kernel.shape.rank == 4:
        kernel = kernel[tf.newaxis, ...]
    if kernel.shape.rank != 5:
        raise ValueError("kernel must have rank 4 or 5.")
    offset_frobenius = tf.sqrt(
        tf.reduce_sum(tf.square(tf.abs(kernel)), axis=[-2, -1])
        + tf.cast(1e-12, kernel.dtype.real_dtype)
    )
    return tf.reduce_sum(offset_frobenius, axis=[1, 2])


def batched_dynamic_conv2d(
    inputs: Tensor,
    kernels: Tensor,
    strides: tuple[int, int] = (1, 1),
    padding: str = "SAME",
) -> Tensor:
    

    inputs = tf.convert_to_tensor(inputs)
    kernels = tf.convert_to_tensor(kernels, dtype=inputs.dtype)
    out_channels = kernels.shape[-1]
    if out_channels is None:
        raise ValueError("The dynamic kernel output-channel dimension must be static.")

    stride_spec = [1, int(strides[0]), int(strides[1]), 1]

    def apply_one(args: tuple[Tensor, Tensor]) -> Tensor:
        x_one, k_one = args
        y_one = tf.nn.conv2d(
            x_one[tf.newaxis, ...],
            k_one,
            strides=stride_spec,
            padding=padding.upper(),
        )
        return y_one[0]

    return tf.map_fn(
        apply_one,
        (inputs, kernels),
        fn_output_signature=tf.TensorSpec(
            shape=(None, None, int(out_channels)), dtype=inputs.dtype
        ),
    )


def _reshape_vector_labels(labels: Tensor) -> Tensor:
    labels = tf.convert_to_tensor(labels)
    if labels.shape.rank == 2:
        return tf.argmax(labels, axis=-1, output_type=tf.int32)
    return tf.cast(tf.reshape(labels, [-1]), tf.int32)


def multimodal_classification_loss(logits: Tensor, labels: Tensor) -> Tensor:
    labels = tf.convert_to_tensor(labels)
    if labels.shape.rank == 2:
        per_example = tf.keras.losses.categorical_crossentropy(
            labels, logits, from_logits=True
        )
    else:
        per_example = tf.keras.losses.sparse_categorical_crossentropy(
            tf.cast(tf.reshape(labels, [-1]), tf.int32), logits, from_logits=True
        )
    return tf.reduce_mean(per_example)


# =============================================================================
# LOT retraction used by ORaW
# =============================================================================


def matrix_inverse_sqrt_newton(
    matrices: Tensor, steps: int = 8, eps: float = 1e-4
) -> Tensor:
    

    matrices = tf.convert_to_tensor(matrices)
    dim = matrices.shape[-1]
    if dim is None:
        raise ValueError("Matrix dimension must be statically known.")
    if matrices.shape[-2] != dim:
        raise ValueError("matrix_inverse_sqrt_newton expects square matrices.")

    complex_or_real_dtype = matrices.dtype
    real_dtype = complex_or_real_dtype.real_dtype
    eps_real = tf.cast(eps, real_dtype)

    eye = tf.eye(
        dim,
        batch_shape=tf.shape(matrices)[:-2],
        dtype=complex_or_real_dtype,
    )
    half = tf.cast(0.5, complex_or_real_dtype)
    three = tf.cast(3.0, complex_or_real_dtype)

    # Numerical Hermitian projection plus positive diagonal regularization.
    hermitian = half * (matrices + tf.linalg.adjoint(matrices))
    hermitian = hermitian + tf.cast(eps_real, complex_or_real_dtype) * eye

    # Real Frobenius norm even when the Fourier-domain matrix is complex.
    abs_hermitian = tf.abs(hermitian)  # complex -> corresponding real dtype
    fro_sq = tf.reduce_sum(
        tf.square(abs_hermitian),
        axis=[-2, -1],
        keepdims=True,
    )
    fro = tf.sqrt(tf.maximum(fro_sq, tf.square(eps_real)))

    y = hermitian / tf.cast(fro, complex_or_real_dtype)
    z = eye
    three_eye = three * eye

    for _ in range(int(steps)):
        update = half * (three_eye - tf.matmul(z, y))
        y = tf.matmul(y, update)
        z = tf.matmul(update, z)

    return z / tf.cast(tf.sqrt(fro), complex_or_real_dtype)


def lot_retract_kernel(
    latent_kernel: Tensor, newton_steps: int = 8, eps: float = 1e-4
) -> Tensor:
    

    latent_kernel = tf.convert_to_tensor(latent_kernel)
    kh, kw, cin, cout = latent_kernel.shape
    if None in (kh, kw, cin, cout):
        raise ValueError("LOT kernel dimensions must be statically known.")

    frequency_input = tf.transpose(latent_kernel, [3, 2, 0, 1])
    frequency_input = tf.signal.fft2d(tf.cast(frequency_input, tf.complex64))
    frequency_input = tf.transpose(frequency_input, [2, 3, 0, 1])

    if int(cout) <= int(cin):
        gram = tf.matmul(frequency_input, frequency_input, adjoint_b=True)
        inv_sqrt = matrix_inverse_sqrt_newton(gram, newton_steps, eps)
        retracted = tf.matmul(inv_sqrt, frequency_input)
    else:
        gram = tf.matmul(frequency_input, frequency_input, adjoint_a=True)
        inv_sqrt = matrix_inverse_sqrt_newton(gram, newton_steps, eps)
        retracted = tf.matmul(frequency_input, inv_sqrt)

    spatial = tf.signal.ifft2d(tf.transpose(retracted, [2, 3, 0, 1]))
    spatial = tf.transpose(spatial, [2, 3, 1, 0])
    return tf.cast(tf.math.real(spatial), latent_kernel.dtype)


# =============================================================================
# Euclidean and Lorentz-hyperbolic SHANW attention
# =============================================================================


def global_pool_descriptors(x: Tensor) -> tuple[Tensor, Tensor, Tensor, Tensor]:
    return (
        tf.reduce_mean(x, axis=[1, 2]),
        tf.reduce_max(x, axis=[1, 2]),
        tf.reduce_sum(x, axis=[1, 2]),
        tf.reduce_min(x, axis=[1, 2]),
    )


def adaptive_average_pool2d(
    x: Tensor,
    output_size: int | tuple[int, int],
) -> Tensor:
    

    x = tf.convert_to_tensor(x)
    if x.shape.rank != 4:
        raise ValueError(
            "adaptive_average_pool2d expects an NHWC rank-4 tensor."
        )

    if isinstance(output_size, int):
        out_h = out_w = int(output_size)
    else:
        out_h, out_w = (int(output_size[0]), int(output_size[1]))
    if out_h <= 0 or out_w <= 0:
        raise ValueError("Adaptive-pooling output dimensions must be positive.")

    input_dtype = x.dtype
    if not input_dtype.is_floating:
        x = tf.cast(x, tf.float32)
        work_dtype = tf.float32
    elif input_dtype in (tf.float16, tf.bfloat16):
        # Accumulate in float32 for numerical stability under mixed precision.
        x = tf.cast(x, tf.float32)
        work_dtype = tf.float32
    else:
        work_dtype = input_dtype

    shape = tf.shape(x)
    height = shape[1]
    width = shape[2]
    checks = [
        tf.debugging.assert_positive(
            height, message="Adaptive pooling received zero input height."
        ),
        tf.debugging.assert_positive(
            width, message="Adaptive pooling received zero input width."
        ),
    ]
    with tf.control_dependencies(checks):
        height = tf.identity(height)
        width = tf.identity(width)

    out_h_t = tf.constant(out_h, dtype=tf.int32)
    out_w_t = tf.constant(out_w, dtype=tf.int32)
    h_index = tf.range(out_h, dtype=tf.int32)
    w_index = tf.range(out_w, dtype=tf.int32)

    # floor(i * H / H_out)
    h_start = tf.math.floordiv(h_index * height, out_h_t)
    w_start = tf.math.floordiv(w_index * width, out_w_t)

    # ceil((i + 1) * H / H_out), using integer arithmetic.
    h_end = tf.math.floordiv(
        (h_index + 1) * height + out_h_t - 1,
        out_h_t,
    )
    w_end = tf.math.floordiv(
        (w_index + 1) * width + out_w_t - 1,
        out_w_t,
    )

    # Integral image I has an explicit zero row/column, so rectangle sums are
    # I[h1,w1] - I[h0,w1] - I[h1,w0] + I[h0,w0].
    integral = tf.cumsum(tf.cumsum(x, axis=1), axis=2)
    integral = tf.pad(
        integral,
        paddings=[[0, 0], [1, 0], [1, 0], [0, 0]],
    )

    def gather_corners(rows: Tensor, cols: Tensor) -> Tensor:
        gathered_rows = tf.gather(integral, rows, axis=1)
        return tf.gather(gathered_rows, cols, axis=2)

    bottom_right = gather_corners(h_end, w_end)
    top_right = gather_corners(h_start, w_end)
    bottom_left = gather_corners(h_end, w_start)
    top_left = gather_corners(h_start, w_start)
    sums = bottom_right - top_right - bottom_left + top_left

    bin_heights = h_end - h_start
    bin_widths = w_end - w_start
    areas = tf.cast(
        bin_heights[:, tf.newaxis] * bin_widths[tf.newaxis, :],
        work_dtype,
    )
    pooled = tf.math.divide_no_nan(
        sums,
        areas[tf.newaxis, :, :, tf.newaxis],
    )

    if input_dtype.is_floating and pooled.dtype != input_dtype:
        pooled = tf.cast(pooled, input_dtype)

    return tf.ensure_shape(
        pooled,
        [None, out_h, out_w, x.shape[-1]],
    )


def lorentz_exp_origin(v: Tensor, curvature: float, eps: float = 1e-7) -> Tensor:
    """Exponential map at the Lorentz origin for tangent vector (0,v)."""

    v = tf.convert_to_tensor(v)
    kappa = tf.cast(curvature, v.dtype)
    sqrt_kappa = tf.sqrt(kappa)
    norm = tf.sqrt(tf.reduce_sum(tf.square(v), axis=-1, keepdims=True) + eps)
    scaled_norm = sqrt_kappa * norm
    time = tf.cosh(scaled_norm) / sqrt_kappa
    coefficient = tf.math.divide_no_nan(tf.sinh(scaled_norm), scaled_norm)
    spatial = coefficient * v
    return tf.concat([time, spatial], axis=-1)


def lorentz_distance_to_prototypes(
    points: Tensor, prototypes: Tensor, curvature: float, eps: float = 1e-6
) -> Tensor:
    """Pairwise Lorentz geodesic distance to channel prototypes.

    ``points`` has shape [..., D+1] and ``prototypes`` has shape [C, D+1].
    The result has shape [..., C].
    """

    points = tf.convert_to_tensor(points)
    prototypes = tf.convert_to_tensor(prototypes, dtype=points.dtype)
    kappa = tf.cast(curvature, points.dtype)

    temporal = -points[..., 0, tf.newaxis] * prototypes[:, 0]
    spatial = tf.einsum("...d,cd->...c", points[..., 1:], prototypes[:, 1:])
    inner = temporal + spatial
    argument = tf.maximum(-kappa * inner, tf.cast(1.0 + eps, points.dtype))
    return tf.acosh(argument) / tf.sqrt(kappa)


class HybridSpaceWeightAttention(layers.Layer):
    """Euclidean--hyperbolic attention used by SHANW.

    One stochastic HyWeCS kernel is shared by the mini-batch. This follows the
    legacy notebook's batch-conditioned weight realization and avoids creating
    a full ``[B, k, k, Cin, Cout]`` kernel bank. At inference with batch size
    one, the realization is sample-conditioned.

    The Euclidean local response is evaluated after adaptive average pooling.
    This is algebraically equivalent to applying the linear 1x1 convolution
    before pooling, because channel-linear projection and average pooling
    commute, while reducing its spatial cost from H*W to k*k.
    """

    def __init__(
        self,
        kernel_size: tuple[int, int],
        out_channels: int,
        curvature: float,
        hyperbolic_dim: int,
        channel_bottleneck_ratio: int,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self.kernel_size = tuple(int(v) for v in kernel_size)
        self.out_channels = int(out_channels)
        self.curvature = float(curvature)
        self.hyperbolic_dim = int(hyperbolic_dim)
        self.channel_bottleneck_ratio = int(channel_bottleneck_ratio)

        self.local_e: Optional[layers.Conv2D] = None
        self.e_desc_in: list[layers.Dense] = []
        self.e_desc_out: Optional[layers.Dense] = None
        self.local_h_projection: Optional[layers.Conv2D] = None
        self.h_desc_projection: list[layers.Dense] = []

    def build(self, input_shape: tf.TensorShape) -> None:
        input_shape = tf.TensorShape(input_shape)
        in_channels = int(input_shape[-1])
        bottleneck = max(1, in_channels // self.channel_bottleneck_ratio)

        self.local_e = layers.Conv2D(
            in_channels,
            kernel_size=1,
            padding="same",
            use_bias=True,
            name="euclidean_local_1x1",
        )
        self.e_desc_in = [
            layers.Dense(bottleneck, activation="relu", name=f"e_desc_{i}_in")
            for i in range(4)
        ]
        self.e_desc_out = layers.Dense(
            in_channels, activation=None, name="euclidean_channel_out"
        )

        self.local_h_projection = layers.Conv2D(
            self.hyperbolic_dim,
            kernel_size=1,
            padding="same",
            use_bias=True,
            name="hyperbolic_local_1x1",
        )
        self.h_desc_projection = [
            layers.Dense(self.hyperbolic_dim, activation=None, name=f"h_desc_{i}")
            for i in range(4)
        ]

        pooled_shape = tf.TensorShape(
            [input_shape[0], self.kernel_size[0], self.kernel_size[1], in_channels]
        )
        descriptor_shape = tf.TensorShape([input_shape[0], in_channels])
        self.local_e.build(pooled_shape)
        for layer in self.e_desc_in:
            layer.build(descriptor_shape)
        self.e_desc_out.build(tf.TensorShape([input_shape[0], bottleneck]))
        self.local_h_projection.build(input_shape)
        for layer in self.h_desc_projection:
            layer.build(descriptor_shape)

        self.h_local_prototypes = self.add_weight(
            name="h_local_prototypes",
            shape=(in_channels, self.hyperbolic_dim),
            initializer="glorot_uniform",
            trainable=True,
        )
        self.h_local_bias = self.add_weight(
            name="h_local_bias",
            shape=(in_channels,),
            initializer="zeros",
            trainable=True,
        )
        self.h_local_beta_logit = self.add_weight(
            name="h_local_beta_logit",
            shape=(in_channels,),
            initializer="zeros",
            trainable=True,
        )
        self.h_channel_prototypes = self.add_weight(
            name="h_channel_prototypes",
            shape=(4, in_channels, self.hyperbolic_dim),
            initializer="glorot_uniform",
            trainable=True,
        )
        self.h_channel_bias = self.add_weight(
            name="h_channel_bias",
            shape=(4, in_channels),
            initializer="zeros",
            trainable=True,
        )
        self.h_channel_gamma_logit = self.add_weight(
            name="h_channel_gamma_logit",
            shape=(4, in_channels),
            initializer="zeros",
            trainable=True,
        )
        super().build(input_shape)

    def _batch_reference(self, x: Tensor) -> Tensor:
        # One feature-conditioned stochastic kernel per mini-batch. The batch
        # mean preserves the original notebook's shared-kernel execution.
        return tf.reduce_mean(x, axis=0, keepdims=True)

    def _euclidean_attention(self, x_ref: Tensor) -> Tensor:
        assert self.local_e is not None and self.e_desc_out is not None
        pooled_x = adaptive_average_pool2d(x_ref, self.kernel_size)
        local = self.local_e(pooled_x)

        pooled = global_pool_descriptors(x_ref)
        hidden = [layer(value) for layer, value in zip(self.e_desc_in, pooled)]
        channel = self.e_desc_out(tf.add_n(hidden))
        channel = channel[:, tf.newaxis, tf.newaxis, :]
        return tf.sigmoid(local + channel)

    def _hyperbolic_attention(self, x_ref: Tensor) -> Tensor:
        assert self.local_h_projection is not None
        local_tangent = tf.tanh(self.local_h_projection(x_ref))
        local_points = lorentz_exp_origin(local_tangent, self.curvature)
        local_proto = lorentz_exp_origin(
            tf.tanh(self.h_local_prototypes), self.curvature
        )
        local_distance = lorentz_distance_to_prototypes(
            local_points, local_proto, self.curvature
        )
        beta = tf.nn.softplus(self.h_local_beta_logit)
        local_logits = self.h_local_bias - beta * tf.square(local_distance)
        local_logits = adaptive_average_pool2d(local_logits, self.kernel_size)

        pooled = global_pool_descriptors(x_ref)
        channel_terms = []
        for index, descriptor in enumerate(pooled):
            tangent = tf.tanh(self.h_desc_projection[index](descriptor))
            point = lorentz_exp_origin(tangent, self.curvature)
            proto = lorentz_exp_origin(
                tf.tanh(self.h_channel_prototypes[index]), self.curvature
            )
            distance = lorentz_distance_to_prototypes(
                point, proto, self.curvature
            )
            gamma = tf.nn.softplus(self.h_channel_gamma_logit[index])
            channel_terms.append(
                self.h_channel_bias[index] - gamma * tf.square(distance)
            )
        channel_logits = tf.add_n(channel_terms)[:, tf.newaxis, tf.newaxis, :]
        return tf.sigmoid(local_logits + channel_logits)

    def call(self, x: Tensor) -> Tensor:
        x_ref = self._batch_reference(tf.convert_to_tensor(x))
        euclidean = self._euclidean_attention(x_ref)[0]
        hyperbolic = self._hyperbolic_attention(x_ref)[0]
        # [2, kh, kw, Cin]. Output-filter broadcasting is deferred until the
        # SHANW recursion, avoiding a large persistent attention tensor.
        return tf.stack([euclidean, hyperbolic], axis=0)


# =============================================================================
# HyWeCS convolution, residual block, and reciprocal fusion
# =============================================================================


class HyWeCSConv2D(layers.Layer):
    """One HyWeCS convolution with exactly two within-filter partitions.

    Each output filter contains exactly ``n = kh * kw * Cin`` trainable base
    coefficients, split into two and only two disjoint partitions:

    * ``oraw_base_values`` has shape ``[Cout, r]``;
    * ``shanw_base_values`` has shape ``[Cout, n-r]``.

    Their sizes add to ``Cout * n``, exactly the number of coefficients in one
    standard convolution kernel. No full ORaW kernel, full SHANW kernel, or
    duplicated base kernel is stored as a trainable variable. A full
    ``[kh,kw,Cin,Cout]`` tensor is created only transiently when the two
    realized partitions are scattered into their prescribed positions before
    convolution and shared Lipschitz calibration.
    """

    def __init__(
        self,
        config: HyWeCSLayerConfig,
        activation: bool = False,
        **kwargs: Any,
    ) -> None:
        config.validate()
        super().__init__(**kwargs)
        self.config = config
        self.activation_enabled = bool(activation)
        self.kernel_size = config.normalized_kernel_size()
        self.strides = config.normalized_strides()
        self.padding = config.padding.upper()

        self.in_channels: Optional[int] = None
        self.n_weights_per_filter: Optional[int] = None
        self.r_weights_per_filter: Optional[int] = None
        self.shanw_weights_per_filter: Optional[int] = None

        # Graph-independent allocation metadata. Tensor copies are materialized
        # locally in each graph, avoiding Keras scratch-graph scope leakage.
        self._oraw_indices_np: Optional[np.ndarray] = None
        self._shanw_indices_np: Optional[np.ndarray] = None
        self._profile_data_path = False
        self._use_cached_oraw = False
        self._cached_oraw_values: Optional[Tensor] = None

        self.attention = HybridSpaceWeightAttention(
            kernel_size=self.kernel_size,
            out_channels=config.filters,
            curvature=config.curvature,
            hyperbolic_dim=config.hyperbolic_dim,
            channel_bottleneck_ratio=config.channel_bottleneck_ratio,
            name="hybrid_space_attention",
        )

    @property
    def oraw_indices(self) -> Tensor:
        if self._oraw_indices_np is None:
            raise RuntimeError(f"Layer {self.name!r} has not been built.")
        return tf.convert_to_tensor(self._oraw_indices_np, dtype=tf.int32)

    @property
    def shanw_indices(self) -> Tensor:
        if self._shanw_indices_np is None:
            raise RuntimeError(f"Layer {self.name!r} has not been built.")
        return tf.convert_to_tensor(self._shanw_indices_np, dtype=tf.int32)

    @property
    def certificate_ready(self) -> bool:
        return self.config.calibration_mode == "verified"

    def build(self, input_shape: tf.TensorShape) -> None:
        input_shape = tf.TensorShape(input_shape)
        self.in_channels = int(input_shape[-1])
        kh, kw = self.kernel_size
        cout = int(self.config.filters)
        n = int(kh * kw * self.in_channels)
        self.n_weights_per_filter = n

        oraw_indices, shanw_indices = make_fixed_within_filter_indices(
            self.kernel_size,
            self.in_channels,
            cout,
            self.config.oraw_fraction,
            self.config.allocation_seed,
        )
        self._oraw_indices_np = np.asarray(oraw_indices, dtype=np.int32)
        self._shanw_indices_np = np.asarray(shanw_indices, dtype=np.int32)
        self.r_weights_per_filter = int(oraw_indices.shape[1])
        self.shanw_weights_per_filter = int(shanw_indices.shape[1])

        if self.r_weights_per_filter + self.shanw_weights_per_filter != n:
            raise RuntimeError("ORaW and SHANW partitions must sum exactly to n.")

        # The two base partitions together contain exactly n coefficients per
        # output filter. The initializer uses the full convolution fan-in n.
        stddev = math.sqrt(2.0 / float(n))
        self.oraw_base_values = self.add_weight(
            name="oraw_base_values",
            shape=(cout, self.r_weights_per_filter),
            initializer=tf.keras.initializers.RandomNormal(
                mean=0.0,
                stddev=stddev,
                seed=self.config.allocation_seed + 17,
            ),
            trainable=True,
        )
        self.shanw_base_values = self.add_weight(
            name="shanw_base_values",
            shape=(cout, self.shanw_weights_per_filter),
            initializer=tf.keras.initializers.RandomNormal(
                mean=0.0,
                stddev=stddev,
                seed=self.config.allocation_seed + 29,
            ),
            trainable=True,
        )

        # These are scalar stochastic-control parameters, not kernel tensors.
        self.geometry_noise_scale = self.add_weight(
            name="geometry_noise_scale",
            shape=(2,),
            initializer=tf.keras.initializers.Constant(self.config.pi_init),
            trainable=True,
        )
        self.rho_logit = self.add_weight(
            name="rho_logit",
            shape=(),
            initializer=tf.keras.initializers.Constant(self.config.rho_logit_init),
            trainable=True,
        )

        if self.config.use_bias:
            self.bias = self.add_weight(
                name="bias",
                shape=(cout,),
                initializer="zeros",
                trainable=True,
            )
        else:
            self.bias = None

        if not self.attention.built:
            self.attention.build(input_shape)
        super().build(input_shape)

    def _single_seed(
        self,
        stochastic_seed: Optional[Tensor | Sequence[int]],
        salt: int,
    ) -> Tensor:
        if stochastic_seed is None:
            seed = tf.random.uniform(
                [2],
                minval=0,
                maxval=np.iinfo(np.int32).max,
                dtype=tf.int32,
            )
        else:
            seed = tf.convert_to_tensor(stochastic_seed, dtype=tf.int32)
            if seed.shape.rank == 2:
                seed = seed[0]
            if seed.shape.rank != 1 or seed.shape[-1] != 2:
                raise ValueError("stochastic_seed must have shape [2] or [B,2].")
        return tf.random.experimental.stateless_fold_in(seed, int(salt))

    def _scatter_partition(
        self,
        values: Tensor,
        flat_indices: Tensor,
    ) -> Tensor:
        """Scatter one ``[Cout,p]`` partition into a temporary full kernel."""

        if self.in_channels is None or self.n_weights_per_filter is None:
            raise RuntimeError("Layer must be built before partition assembly.")
        values = tf.convert_to_tensor(values)
        flat_indices = tf.convert_to_tensor(flat_indices, dtype=tf.int32)
        cout = int(self.config.filters)
        kh, kw = self.kernel_size

        output_ids = tf.broadcast_to(
            tf.range(cout, dtype=tf.int32)[:, tf.newaxis],
            tf.shape(flat_indices),
        )
        scatter_indices = tf.stack([flat_indices, output_ids], axis=-1)
        flat_kernel = tf.scatter_nd(
            tf.reshape(scatter_indices, [-1, 2]),
            tf.reshape(values, [-1]),
            tf.constant([self.n_weights_per_filter, cout], dtype=tf.int32),
        )
        return tf.reshape(flat_kernel, [kh, kw, self.in_channels, cout])

    def _gather_partition(
        self,
        kernel: Tensor,
        flat_indices: Tensor,
    ) -> Tensor:
        """Gather a ``[Cout,p]`` partition from a temporary full kernel."""

        if self.n_weights_per_filter is None:
            raise RuntimeError("Layer must be built before partition gathering.")
        cout = int(self.config.filters)
        flat = tf.reshape(kernel, [self.n_weights_per_filter, cout])
        flat = tf.transpose(flat, [1, 0])  # [Cout,n]
        return tf.gather(
            flat,
            tf.convert_to_tensor(flat_indices, dtype=tf.int32),
            axis=1,
            batch_dims=1,
        )

    def _sample_oraw_values(
        self,
        stochastic_seed: Optional[Tensor | Sequence[int]],
        dtype: tf.dtypes.DType,
    ) -> Tensor:
        """Realize only the r ORaW coefficients of every filter."""

        if self.n_weights_per_filter is None:
            raise RuntimeError("Layer must be built before ORaW realization.")
        base = tf.cast(self.oraw_base_values, dtype)

        if self._profile_data_path:
            return base
        if self._use_cached_oraw:
            if self._cached_oraw_values is None:
                raise RuntimeError(
                    "ORaW cache was enabled before refresh_oraw_cache()."
                )
            return tf.cast(self._cached_oraw_values, dtype)

        seed = self._single_seed(stochastic_seed, self.config.seed_salt + 11)
        alpha = tf.random.stateless_normal(tf.shape(base), seed=seed, dtype=dtype)
        scale = tf.cast(
            self.config.sigma_rp / math.sqrt(self.n_weights_per_filter),
            dtype,
        )
        latent_values = base + scale * alpha

        # LOT is applied to the temporary ORaW candidate tensor, after which
        # only the prescribed r coordinates are retained, matching the paper's
        # restriction T_LOT(.)|_{I_oraw}. This tensor is not a trainable kernel.
        latent_candidate = self._scatter_partition(
            latent_values,
            self.oraw_indices,
        )
        retracted_candidate = lot_retract_kernel(
            latent_candidate,
            newton_steps=self.config.lot_newton_steps,
            eps=self.config.lot_eps,
        )
        return self._gather_partition(retracted_candidate, self.oraw_indices)

    def refresh_oraw_cache(
        self,
        stochastic_seed: Optional[Tensor | Sequence[int]] = None,
    ) -> None:
        value = self._sample_oraw_values(
            stochastic_seed,
            tf.as_dtype(self.compute_dtype),
        )
        self._cached_oraw_values = tf.stop_gradient(value)

    def set_use_cached_oraw(self, enabled: bool) -> None:
        self._use_cached_oraw = bool(enabled)
        if not self._use_cached_oraw:
            self._cached_oraw_values = None

    def set_profile_data_path(self, enabled: bool) -> None:
        self._profile_data_path = bool(enabled)

    def _sample_shanw_values(
        self,
        x: Tensor,
        stochastic_seed: Optional[Tensor | Sequence[int]],
    ) -> tuple[Tensor, Tensor, Tensor]:
        """Realize only the complementary n-r SHANW coefficients."""

        if self.n_weights_per_filter is None or self.shanw_weights_per_filter is None:
            raise RuntimeError("Layer must be built before SHANW realization.")
        dtype = x.dtype
        cout = int(self.config.filters)
        d = int(self.shanw_weights_per_filter)

        # Attention is first learned over the n flattened positions and then
        # restricted to the n-r SHANW positions of each output filter.
        attention_full = self.attention(x)  # [2,kh,kw,Cin]
        attention_full = tf.reshape(
            attention_full,
            [2, self.n_weights_per_filter],
        )
        attention = tf.gather(
            attention_full,
            self.shanw_indices,
            axis=1,
        )  # [2,Cout,d]

        product = tf.ones((cout, d), dtype=dtype)
        pi = tf.cast(self.geometry_noise_scale, dtype)[:, tf.newaxis, tf.newaxis]
        lambda_maxima = []

        for recursion in range(int(self.config.shanw_recursions)):
            seed = self._single_seed(
                stochastic_seed,
                self.config.seed_salt + 101 + recursion,
            )
            eta = tf.random.stateless_normal(
                (2, cout, d),
                seed=seed,
                dtype=dtype,
            )
            self_modulated = eta * (pi + eta * pi)
            lambda_maxima.append(tf.reduce_max(tf.abs(self_modulated)))
            geometry_logits = attention * self_modulated
            geometry_weights = tf.nn.softmax(geometry_logits, axis=0)
            fused = tf.reduce_sum(geometry_weights * attention, axis=0)
            product = product * tf.sigmoid(fused)

        rho = tf.nn.softplus(tf.cast(self.rho_logit, dtype))
        shanw_values = tf.cast(self.shanw_base_values, dtype) + rho * product
        return shanw_values, tf.stack(lambda_maxima), attention

    def _calibrate(
        self,
        oraw_values: Tensor,
        shanw_values: Tensor,
        lambda_max: Tensor,
    ) -> tuple[Tensor, Mapping[str, Tensor]]:
        """Assemble the two partitions once and calibrate the full mapping."""

        dtype = oraw_values.dtype
        oraw_kernel = self._scatter_partition(oraw_values, self.oraw_indices)
        shanw_kernel = self._scatter_partition(shanw_values, self.shanw_indices)
        combined = oraw_kernel + shanw_kernel
        tau = tf.cast(self.config.tau, dtype)

        if self.config.calibration_mode == "verified":
            rho = tf.nn.softplus(tf.cast(self.rho_logit, dtype))
            update_bound = rho * tf.sigmoid(tf.cast(1.0, dtype)) ** int(
                self.config.shanw_recursions
            )
            shanw_abs_values = (
                tf.abs(tf.cast(self.shanw_base_values, dtype)) + update_bound
            )
            shanw_abs_kernel = self._scatter_partition(
                shanw_abs_values,
                self.shanw_indices,
            )
            static_bound = (
                convolution_operator_upper_bound(oraw_kernel[tf.newaxis, ...])[0]
                + convolution_operator_upper_bound(
                    shanw_abs_kernel[tf.newaxis, ...]
                )[0]
            )
            rho_term = (
                rho
                * tf.cast(self.config.attention_lipschitz_bound, dtype)
                / tf.cast(4.0, dtype)
            )
            sensitivity_sum = tf.reduce_sum(
                1.0 + lambda_max / tf.sqrt(tf.cast(2.0, dtype))
            )
            weight_lipschitz = rho_term * sensitivity_sum
            verified_bound = static_bound + tf.cast(
                self.config.feature_norm_bound,
                dtype,
            ) * weight_lipschitz
        else:
            static_bound = convolution_operator_upper_bound(
                combined[tf.newaxis, ...]
            )[0]
            weight_lipschitz = tf.cast(0.0, dtype)
            verified_bound = static_bound

        scale = tf.minimum(
            tf.cast(1.0, dtype),
            tau / tf.maximum(verified_bound, tf.cast(1e-8, dtype)),
        )
        calibrated = combined * scale
        diagnostics = {
            "scale": scale,
            "precalibration_bound": verified_bound,
            "fixed_kernel_bound": static_bound,
            "weight_lipschitz_bound": weight_lipschitz,
            "oraw_partition": oraw_values,
            "shanw_partition": shanw_values,
            "oraw_kernel": oraw_kernel,
            "shanw_kernel": shanw_kernel,
        }
        return calibrated, diagnostics

    def realize_kernel(
        self,
        x: Tensor,
        stochastic_seed: Optional[Tensor | Sequence[int]] = None,
    ) -> tuple[Tensor, Mapping[str, Tensor]]:
        x = tf.convert_to_tensor(x)
        oraw_values = self._sample_oraw_values(stochastic_seed, x.dtype)
        shanw_values, lambda_max, attention = self._sample_shanw_values(
            x,
            stochastic_seed,
        )
        calibrated, diagnostics = self._calibrate(
            oraw_values,
            shanw_values,
            lambda_max,
        )
        diagnostics = dict(diagnostics)
        diagnostics.update(
            {
                "hybrid_attention": attention,
                "lambda_max": lambda_max,
            }
        )
        return calibrated, diagnostics

    def apply_kernel(self, x: Tensor, kernel: Tensor) -> Tensor:
        y = tf.nn.conv2d(
            x,
            tf.cast(kernel, x.dtype),
            strides=[1, self.strides[0], self.strides[1], 1],
            padding=self.padding,
        )
        if self.bias is not None:
            y = tf.nn.bias_add(y, tf.cast(self.bias, y.dtype))
        if self.activation_enabled:
            y = tf.nn.relu(y)
        return y

    def call(
        self,
        x: Tensor,
        training: Optional[bool] = None,
        stochastic_seed: Optional[Tensor | Sequence[int]] = None,
        return_kernel: bool = False,
        return_diagnostics: bool = False,
    ) -> Any:
        del training
        kernel, diagnostics = self.realize_kernel(x, stochastic_seed)
        y = self.apply_kernel(x, kernel)
        if return_kernel and return_diagnostics:
            return y, kernel, diagnostics
        if return_kernel:
            return y, kernel
        if return_diagnostics:
            return y, diagnostics
        return y

    def get_config(self) -> dict[str, Any]:
        config = super().get_config()
        config.update(
            {
                "config": self.config.__dict__,
                "activation": self.activation_enabled,
            }
        )
        return config

class NonExpansiveShortcut(layers.Layer):
    """Average-pool and channel pad/slice shortcut without trainable weights."""

    def __init__(self, out_channels: int, stride: int = 1, **kwargs: Any) -> None:
        super().__init__(**kwargs)
        self.out_channels = int(out_channels)
        self.stride = int(stride)
        self.in_channels: Optional[int] = None

    def build(self, input_shape: tf.TensorShape) -> None:
        self.in_channels = int(input_shape[-1])
        super().build(input_shape)

    def call(self, x: Tensor) -> Tensor:
        if self.stride > 1:
            x = tf.nn.avg_pool2d(
                x,
                ksize=self.stride,
                strides=self.stride,
                padding="SAME",
            )
        assert self.in_channels is not None
        if self.in_channels < self.out_channels:
            pad = self.out_channels - self.in_channels
            left = pad // 2
            right = pad - left
            x = tf.pad(x, [[0, 0], [0, 0], [0, 0], [left, right]])
        elif self.in_channels > self.out_channels:
            x = x[..., : self.out_channels]
        return x


class HyWeCSResidualBlock(layers.Layer):
    """Two-layer residual mapping from the current paper."""

    def __init__(
        self,
        filters: int,
        stride: int,
        base_config: HyWeCSLayerConfig,
        seed_salt: int,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        conv1_config = replace(
            base_config,
            filters=filters,
            strides=stride,
            seed_salt=seed_salt + 1,
            allocation_seed=base_config.allocation_seed + seed_salt + 1,
        )
        conv2_config = replace(
            base_config,
            filters=filters,
            strides=1,
            seed_salt=seed_salt + 2,
            allocation_seed=base_config.allocation_seed + seed_salt + 2,
        )
        self.conv1 = HyWeCSConv2D(conv1_config, activation=False, name="conv1")
        self.conv2 = HyWeCSConv2D(conv2_config, activation=False, name="conv2")
        self.shortcut = NonExpansiveShortcut(filters, stride, name="shortcut")
        self.seed_salt = int(seed_salt)
        self.filters = int(filters)
        self.stride = int(stride)

    def build(self, input_shape: tf.TensorShape) -> None:
        input_shape = tf.TensorShape(input_shape)
        if not self.conv1.built:
            self.conv1.build(input_shape)
        hidden_shape = same_padding_conv_output_shape(
            input_shape, self.filters, self.stride
        )
        if not self.conv2.built:
            self.conv2.build(hidden_shape)
        if not self.shortcut.built:
            self.shortcut.build(input_shape)
        super().build(input_shape)

    @property
    def certificate_ready(self) -> bool:
        return self.conv1.certificate_ready and self.conv2.certificate_ready

    def call(
        self,
        x: Tensor,
        training: Optional[bool] = None,
        stochastic_seed: Optional[Tensor | Sequence[int]] = None,
    ) -> Tensor:
        batch_seeds = canonicalize_seed_batch(stochastic_seed, tf.shape(x)[0])
        y = self.conv1(
            x,
            training=training,
            stochastic_seed=fold_in_seed_batch(batch_seeds, self.seed_salt + 1),
        )
        y = tf.nn.relu(y)
        y = self.conv2(
            y,
            training=training,
            stochastic_seed=fold_in_seed_batch(batch_seeds, self.seed_salt + 2),
        )
        return self.shortcut(x) + y


class HyWeCSEncoderStage(layers.Layer):
    """Residual stage plus channel-preserving 1x1 terminal HyWeCS weights."""

    def __init__(
        self,
        filters: int,
        num_blocks: int,
        first_stride: int,
        base_config: HyWeCSLayerConfig,
        seed_salt: int,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self.blocks = []
        for index in range(int(num_blocks)):
            block = HyWeCSResidualBlock(
                filters=filters,
                stride=first_stride if index == 0 else 1,
                base_config=base_config,
                seed_salt=seed_salt + 20 * index,
                name=f"block_{index}",
            )
            self.blocks.append(block)

        terminal_config = replace(
            base_config,
            filters=filters,
            kernel_size=1,
            strides=1,
            seed_salt=seed_salt + 997,
            allocation_seed=base_config.allocation_seed + seed_salt + 997,
        )
        self.terminal = HyWeCSConv2D(
            terminal_config, activation=False, name="terminal_1x1"
        )
        self.seed_salt = int(seed_salt)
        self.filters = int(filters)
        self.first_stride = int(first_stride)
        self._output_shape_spec: Optional[tf.TensorShape] = None

    @property
    def output_shape_spec(self) -> tf.TensorShape:
        if self._output_shape_spec is None:
            raise RuntimeError(f"Stage {self.name!r} has not been built.")
        return self._output_shape_spec

    def build(self, input_shape: tf.TensorShape) -> None:
        current_shape = tf.TensorShape(input_shape)
        for block in self.blocks:
            if not block.built:
                block.build(current_shape)
            current_shape = same_padding_conv_output_shape(
                current_shape, block.filters, block.stride
            )
        if not self.terminal.built:
            self.terminal.build(current_shape)
        self._output_shape_spec = current_shape
        super().build(input_shape)

    @property
    def certificate_ready(self) -> bool:
        return all(block.certificate_ready for block in self.blocks) and \
            self.terminal.certificate_ready

    def call(
        self,
        x: Tensor,
        training: Optional[bool] = None,
        stochastic_seed: Optional[Tensor | Sequence[int]] = None,
    ) -> tuple[Tensor, Tensor, Mapping[str, Tensor]]:
        seeds = canonicalize_seed_batch(stochastic_seed, tf.shape(x)[0])
        for index, block in enumerate(self.blocks):
            x = block(
                x,
                training=training,
                stochastic_seed=fold_in_seed_batch(
                    seeds, self.seed_salt + 20 * index
                ),
            )

        # Eq. (6) reuses the terminal kernel for the opposite modality; no
        # redundant terminal convolution is evaluated on the source feature.
        terminal_kernel, diagnostics = self.terminal.realize_kernel(
            x,
            stochastic_seed=fold_in_seed_batch(seeds, self.seed_salt + 997),
        )
        return x, terminal_kernel, diagnostics


class GlobalScoreCalibration(layers.Layer):
    def __init__(
        self,
        target: float = 2.0,
        verified_network_bound: Optional[float] = None,
        **kwargs: Any,
    ) -> None:
        super().__init__(**kwargs)
        self.target = float(target)
        self.verified_network_bound = verified_network_bound

    @property
    def certificate_ready(self) -> bool:
        return (
            self.verified_network_bound is not None
            and self.verified_network_bound > 0.0
        )

    def call(self, logits: Tensor) -> Tensor:
        if self.verified_network_bound is None:
            return logits
        scale = min(1.0, self.target / float(self.verified_network_bound))
        return tf.cast(scale, logits.dtype) * logits


class HyWeCSMultimodalResNet(keras.Model):
    """Two-modality standard-stem ResNet-18 with HyWeCS residual layers."""

    def __init__(self, config: HyWeCSModelConfig, **kwargs: Any) -> None:
        config.validate()
        super().__init__(**kwargs)
        self.paper_config = config

        shared_layer_config = HyWeCSLayerConfig(
            filters=config.base_width,
            kernel_size=3,
            strides=1,
            oraw_fraction=config.oraw_fraction,
            allocation_seed=config.allocation_seed,
            sigma_rp=config.sigma_rp,
            lot_newton_steps=config.lot_newton_steps,
            lot_eps=config.lot_eps,
            shanw_recursions=config.shanw_recursions,
            curvature=config.curvature,
            hyperbolic_dim=config.hyperbolic_dim,
            channel_bottleneck_ratio=config.channel_bottleneck_ratio,
            tau=config.tau,
            calibration_mode=config.calibration_mode,
            feature_norm_bound=config.feature_norm_bound,
            attention_lipschitz_bound=config.attention_lipschitz_bound,
        )

        # HyWeCS replaces convolutions inside residual blocks. A conventional
        # ResNet-18 stem prevents the 128x128 feature map from being retained
        # through the entire first stage, which was a major source of the
        # previous 511G profile.
        self.stem1 = layers.Conv2D(
            config.base_width, 7, strides=2, padding="same", use_bias=False,
            name="stem_modality_1",
        )
        self.stem2 = layers.Conv2D(
            config.base_width, 7, strides=2, padding="same", use_bias=False,
            name="stem_modality_2",
        )
        self.stem_relu1 = layers.ReLU(name="stem_relu_modality_1")
        self.stem_relu2 = layers.ReLU(name="stem_relu_modality_2")
        self.stem_pool1 = layers.MaxPool2D(3, strides=2, padding="same", name="stem_pool_modality_1")
        self.stem_pool2 = layers.MaxPool2D(3, strides=2, padding="same", name="stem_pool_modality_2")

        self.stages1: list[HyWeCSEncoderStage] = []
        self.stages2: list[HyWeCSEncoderStage] = []
        for stage_index, (blocks, multiplier) in enumerate(
            zip(config.blocks_per_stage, config.stage_width_multipliers)
        ):
            filters = config.base_width * multiplier
            stride = 1 if stage_index == 0 else 2
            self.stages1.append(
                HyWeCSEncoderStage(
                    filters, blocks, stride, shared_layer_config,
                    seed_salt=1000 + 200 * stage_index,
                    name=f"modality_1_stage_{stage_index + 1}",
                )
            )
            self.stages2.append(
                HyWeCSEncoderStage(
                    filters, blocks, stride, shared_layer_config,
                    seed_salt=2000 + 200 * stage_index,
                    name=f"modality_2_stage_{stage_index + 1}",
                )
            )

        self.pool = layers.GlobalAveragePooling2D(name="global_pool")
        self.classifier = layers.Dense(
            config.num_classes, activation=None, name="uncalibrated_logits"
        )
        self.global_calibration = GlobalScoreCalibration(
            target=config.global_lipschitz_target,
            verified_network_bound=config.verified_network_bound,
            name="global_score_calibration",
        )

    def _hywecs_layers(self) -> list[HyWeCSConv2D]:
        result: list[HyWeCSConv2D] = []
        for stage in self.stages1 + self.stages2:
            for block in stage.blocks:
                result.extend([block.conv1, block.conv2])
            result.append(stage.terminal)
        return result

    def set_profile_data_path(self, enabled: bool) -> None:
        for layer in self._hywecs_layers():
            layer.set_profile_data_path(enabled)

    def refresh_oraw_caches(
        self, stochastic_seed: Optional[Tensor | Sequence[int]] = None
    ) -> None:
        seed = tf.convert_to_tensor(
            [0, 1] if stochastic_seed is None else stochastic_seed, dtype=tf.int32
        )
        if seed.shape.rank == 2:
            seed = seed[0]
        for index, layer in enumerate(self._hywecs_layers()):
            layer.refresh_oraw_cache(
                tf.random.experimental.stateless_fold_in(seed, index + 1)
            )

    def set_use_cached_oraw(self, enabled: bool) -> None:
        for layer in self._hywecs_layers():
            layer.set_use_cached_oraw(enabled)

    def build(self, input_shape: Sequence[tf.TensorShape]) -> None:
        if not isinstance(input_shape, (tuple, list)) or len(input_shape) != 2:
            raise ValueError("HyWeCSMultimodalResNet.build expects two NHWC shapes.")
        shape1, shape2 = tf.TensorShape(input_shape[0]), tf.TensorShape(input_shape[1])
        if shape1.rank != 4 or shape2.rank != 4:
            raise ValueError("Both modalities must be rank-4 NHWC tensors.")

        self.stem1.build(shape1)
        self.stem2.build(shape2)
        shape1 = same_padding_conv_output_shape(shape1, self.paper_config.base_width, (2, 2))
        shape2 = same_padding_conv_output_shape(shape2, self.paper_config.base_width, (2, 2))
        shape1 = same_padding_conv_output_shape(shape1, self.paper_config.base_width, (2, 2))
        shape2 = same_padding_conv_output_shape(shape2, self.paper_config.base_width, (2, 2))

        for stage1, stage2 in zip(self.stages1, self.stages2):
            stage1.build(shape1)
            stage2.build(shape2)
            shape1, shape2 = stage1.output_shape_spec, stage2.output_shape_spec

        fused_shape = tf.TensorShape(
            [shape1[0], shape1[1], shape1[2], int(shape1[-1]) + int(shape2[-1])]
        )
        self.pool.build(fused_shape)
        self.classifier.build(tf.TensorShape([fused_shape[0], fused_shape[-1]]))
        super().build(input_shape)

    @property
    def certificate_ready(self) -> bool:
        stages_ready = all(s.certificate_ready for s in self.stages1 + self.stages2)
        return (
            self.paper_config.calibration_mode == "verified"
            and stages_ready
            and self.global_calibration.certificate_ready
            and self.paper_config.reciprocal_cross_fusion_verified
        )

    def assert_certificate_ready(self) -> None:
        if not self.certificate_ready:
            raise RuntimeError(
                "Formal certification requires verified layer, cross-fusion, "
                "and network bounds; diagnostic mode is not a certificate."
            )

    def call(
        self,
        inputs: Sequence[Tensor],
        training: Optional[bool] = None,
        stochastic_seed: Optional[Tensor | Sequence[int]] = None,
        return_uncalibrated_logits: bool = False,
        return_stage_diagnostics: bool = False,
    ) -> Any:
        if not isinstance(inputs, (tuple, list)) or len(inputs) != 2:
            raise ValueError("HyWeCSMultimodalResNet expects two modality tensors.")
        x1, x2 = inputs
        tf.debugging.assert_equal(tf.shape(x1)[0], tf.shape(x2)[0])
        seeds = canonicalize_seed_batch(stochastic_seed, tf.shape(x1)[0])

        x1 = self.stem_pool1(self.stem_relu1(self.stem1(x1)))
        x2 = self.stem_pool2(self.stem_relu2(self.stem2(x2)))

        stage_diagnostics = []
        for stage_index, (stage1, stage2) in enumerate(zip(self.stages1, self.stages2)):
            x1_residual, kernel1, diagnostics1 = stage1(
                x1, training=training,
                stochastic_seed=fold_in_seed_batch(seeds, 10000 + stage_index),
            )
            x2_residual, kernel2, diagnostics2 = stage2(
                x2, training=training,
                stochastic_seed=fold_in_seed_batch(seeds, 20000 + stage_index),
            )
            x1 = stage1.terminal.apply_kernel(x2_residual, kernel1)
            x2 = stage2.terminal.apply_kernel(x1_residual, kernel2)
            stage_diagnostics.append((diagnostics1, diagnostics2))

        fused = tf.concat([x1, x2], axis=-1)
        uncalibrated_logits = self.classifier(self.pool(fused))
        logits = self.global_calibration(uncalibrated_logits)

        if return_stage_diagnostics and return_uncalibrated_logits:
            return logits, uncalibrated_logits, stage_diagnostics
        if return_stage_diagnostics:
            return logits, stage_diagnostics
        if return_uncalibrated_logits:
            return logits, uncalibrated_logits
        return logits


# =============================================================================
# Model builder and data helpers
# =============================================================================


def build_hywecs_multimodal_resnet18(config: HyWeCSModelConfig) -> HyWeCSMultimodalResNet:
    model = HyWeCSMultimodalResNet(config, name="hywecs_multimodal_resnet18")
    model.build(
        [
            tf.TensorShape([None, *config.input_shape_modality_1]),
            tf.TensorShape([None, *config.input_shape_modality_2]),
        ]
    )
    dummy1 = tf.zeros([1, *config.input_shape_modality_1], dtype=tf.float32)
    dummy2 = tf.zeros([1, *config.input_shape_modality_2], dtype=tf.float32)
    # Shape/state validation does not need to execute the expensive LOT
    # reparameterization for all 40 HyWeCS layers.
    model.set_profile_data_path(True)
    try:
        _ = model(
            [dummy1, dummy2], training=False,
            stochastic_seed=tf.constant([config.seed, config.seed + 1], tf.int32),
        )
    finally:
        model.set_profile_data_path(False)
    return model


def load_numpy_multimodal_split(directory: str | Path = ".") -> dict[str, np.ndarray]:
    directory = Path(directory)
    names = {
        "X_train_image": "X_train1.npy",
        "X_train_ehr": "X_train2.npy",
        "y_train": "y_train.npy",
        "X_val_image": "X_val1.npy",
        "X_val_ehr": "X_val2.npy",
        "y_val": "y_val.npy",
        "X_test_image": "X_test1.npy",
        "X_test_ehr": "X_test2.npy",
        "y_test": "y_test.npy",
    }
    arrays = {key: np.load(directory / filename) for key, filename in names.items()}
    for split in ("train", "val", "test"):
        counts = {
            len(arrays[f"X_{split}_image"]),
            len(arrays[f"X_{split}_ehr"]),
            len(arrays[f"y_{split}"]),
        }
        if len(counts) != 1:
            raise ValueError(f"Sample-count mismatch in {split} split: {counts}")
    return arrays


def reshape_vector_modality(vectors: np.ndarray, target_shape: Sequence[int]) -> np.ndarray:
    vectors = np.asarray(vectors)
    target_shape = tuple(int(v) for v in target_shape)
    expected = int(np.prod(target_shape))
    if vectors.ndim != 2 or vectors.shape[1] != expected:
        raise ValueError(
            f"Expected [N,{expected}] vectors for target shape {target_shape}; "
            f"received {vectors.shape}. The paper does not define an automatic "
            "padding rule, so the reshape must be exact."
        )
    return vectors.reshape((vectors.shape[0], *target_shape))


def make_tf_dataset(
    x1: np.ndarray,
    x2: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
    seed: int = 0,
) -> tf.data.Dataset:
    dataset = tf.data.Dataset.from_tensor_slices(((x1, x2), y))
    if shuffle:
        dataset = dataset.shuffle(len(y), seed=seed, reshuffle_each_iteration=True)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


# =============================================================================
# Paper-aligned training
# =============================================================================


def add_gaussian_multimodal_noise(
    modalities: Sequence[Tensor], sigma: float, seeds: Tensor
) -> list[Tensor]:
    noisy = []
    for index, x in enumerate(modalities):
        modality_seeds = fold_in_seed_batch(seeds, 500 + index)
        sample_shape = tuple(int(v) for v in x.shape[1:])
        noise = stateless_normal_per_example(modality_seeds, sample_shape, x.dtype)
        noisy.append(x + tf.cast(sigma, x.dtype) * noise)
    return noisy


def certification_compatible_train_step(
    model: HyWeCSMultimodalResNet,
    optimizer: tf.keras.optimizers.Optimizer,
    modalities: Sequence[Tensor],
    labels: Tensor,
    sigma: float,
    q_tr: int = 1,
) -> Mapping[str, Tensor]:
    """Algorithm-2-style training with Gaussian and internal joint randomness."""

    modalities = [tf.convert_to_tensor(x, tf.float32) for x in modalities]
    batch_size = tf.shape(modalities[0])[0]

    with tf.GradientTape() as tape:
        loss = tf.cast(0.0, tf.float32)
        for q in range(int(q_tr)):
            seeds = random_seed_batch(batch_size)
            noisy = add_gaussian_multimodal_noise(modalities, sigma, seeds)
            logits = model(
                noisy,
                training=True,
                stochastic_seed=fold_in_seed_batch(seeds, 1000 + q),
            )
            loss = loss + multimodal_classification_loss(logits, labels) / float(q_tr)

        if model.losses:
            loss = loss + tf.add_n(model.losses)

    gradients = tape.gradient(loss, model.trainable_variables)
    pairs = [(g, v) for g, v in zip(gradients, model.trainable_variables) if g is not None]
    optimizer.apply_gradients(pairs)
    return {"loss": loss}


def eot_pgd_multimodal(
    model: HyWeCSMultimodalResNet,
    modalities: Sequence[Tensor],
    labels: Tensor,
    sigma: float,
    epsilon: float,
    step_size: float,
    steps: int,
    q_attack: int,
    random_start: bool = True,
    clip_values: Optional[Sequence[tuple[float, float] | None]] = None,
) -> list[Tensor]:
    """Fixed-realization EOT-PGD inner maximization from the paper algorithm."""

    clean = [tf.convert_to_tensor(x, tf.float32) for x in modalities]
    batch_size = tf.shape(clean[0])[0]
    if clip_values is None:
        clip_values = [None] * len(clean)
    if len(clip_values) != len(clean):
        raise ValueError("clip_values must match the number of modalities.")

    attack_seeds = [random_seed_batch(batch_size) for _ in range(int(q_attack))]
    # ORaW is input-independent. For the common q_attack=1 training setting,
    # realize it once and reuse it throughout the PGD trajectory, exactly as
    # required by the fixed-realization attack phase. Final EOT evaluation may
    # still use q_attack>1, in which case each realization is recomputed.
    use_oraw_cache = int(q_attack) == 1
    if use_oraw_cache:
        model.refresh_oraw_caches(attack_seeds[0])
        model.set_use_cached_oraw(True)

    fixed_input_noise = [
        add_gaussian_multimodal_noise(
            [tf.zeros_like(x) for x in clean], sigma, attack_seeds[q]
        )
        for q in range(int(q_attack))
    ]

    deltas = []
    for modality_index, x in enumerate(clean):
        if random_start:
            seed = fold_in_seed_batch(attack_seeds[0], 7000 + modality_index)
            shape = tuple(int(v) for v in x.shape[1:])
            unit = stateless_normal_per_example(seed, shape, x.dtype)
            initial = tf.clip_by_value(unit, -1.0, 1.0) * epsilon
        else:
            initial = tf.zeros_like(x)
        deltas.append(tf.Variable(initial, trainable=True))

    for _ in range(int(steps)):
        with tf.GradientTape() as tape:
            for delta in deltas:
                tape.watch(delta)
            loss = tf.cast(0.0, tf.float32)
            for q in range(int(q_attack)):
                attacked = [
                    clean[i] + deltas[i] + fixed_input_noise[q][i]
                    for i in range(len(clean))
                ]
                logits = model(
                    attacked,
                    training=True,
                    stochastic_seed=attack_seeds[q],
                )
                loss = loss + multimodal_classification_loss(logits, labels) / float(q_attack)

        gradients = tape.gradient(loss, deltas)
        for index, (delta, gradient) in enumerate(zip(deltas, gradients)):
            if gradient is None:
                raise RuntimeError(f"Missing attack gradient for modality {index}.")
            delta.assign_add(tf.cast(step_size, delta.dtype) * tf.sign(gradient))
            delta.assign(tf.clip_by_value(delta, -epsilon, epsilon))
            bounds = clip_values[index]
            if bounds is not None:
                lo, hi = bounds
                projected_input = tf.clip_by_value(clean[index] + delta, lo, hi)
                delta.assign(projected_input - clean[index])

    if use_oraw_cache:
        model.set_use_cached_oraw(False)
    return [tf.stop_gradient(delta) for delta in deltas]


def adversarial_train_step(
    model: HyWeCSMultimodalResNet,
    optimizer: tf.keras.optimizers.Optimizer,
    modalities: Sequence[Tensor],
    labels: Tensor,
    sigma: float,
    epsilon: float,
    step_size: float,
    attack_steps: int,
    q_attack: int,
    q_update: int,
    lambda_adv: float,
    clip_values: Optional[Sequence[tuple[float, float] | None]] = None,
) -> Mapping[str, Tensor]:
    """Algorithm-4-style EOT adversarial training with fresh update draws."""

    clean = [tf.convert_to_tensor(x, tf.float32) for x in modalities]
    deltas = eot_pgd_multimodal(
        model,
        clean,
        labels,
        sigma,
        epsilon,
        step_size,
        attack_steps,
        q_attack,
        clip_values=clip_values,
    )
    batch_size = tf.shape(clean[0])[0]

    with tf.GradientTape() as tape:
        clean_loss = tf.cast(0.0, tf.float32)
        adv_loss = tf.cast(0.0, tf.float32)
        for q in range(int(q_update)):
            seeds = random_seed_batch(batch_size)
            gaussian = add_gaussian_multimodal_noise(
                [tf.zeros_like(x) for x in clean], sigma, seeds
            )
            clean_input = [clean[i] + gaussian[i] for i in range(len(clean))]
            adv_input = [clean[i] + deltas[i] + gaussian[i] for i in range(len(clean))]

            clean_logits = model(clean_input, training=True, stochastic_seed=seeds)
            adv_logits = model(adv_input, training=True, stochastic_seed=seeds)
            clean_loss = clean_loss + multimodal_classification_loss(
                clean_logits, labels
            ) / float(q_update)
            adv_loss = adv_loss + multimodal_classification_loss(
                adv_logits, labels
            ) / float(q_update)

        total = clean_loss + tf.cast(lambda_adv, tf.float32) * adv_loss
        if model.losses:
            total = total + tf.add_n(model.losses)

    gradients = tape.gradient(total, model.trainable_variables)
    pairs = [(g, v) for g, v in zip(gradients, model.trainable_variables) if g is not None]
    optimizer.apply_gradients(pairs)
    return {"loss": total, "clean_loss": clean_loss, "adv_loss": adv_loss}


# =============================================================================
# Joint-randomness certification
# =============================================================================


def clopper_pearson_lower(k: int, n: int, alpha: float) -> float:
    if n <= 0 or not 0 <= k <= n:
        raise ValueError("Require n>0 and 0<=k<=n.")
    if k == 0:
        return 0.0
    return float(scipy_beta.ppf(alpha, k, n - k + 1))


def _repeat_one_sample(x: np.ndarray, batch_size: int) -> np.ndarray:
    return np.repeat(np.asarray(x, dtype=np.float32)[None, ...], batch_size, axis=0)


def sample_joint_scores(
    model: HyWeCSMultimodalResNet,
    x_modalities: Sequence[np.ndarray],
    sigma: float,
    num_samples: int,
    batch_size: int = 64,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """Sample iid scores under Gaussian plus complete HyWeCS randomness.

    The efficient training implementation shares one internal kernel across a
    mini-batch. Formal Clopper--Pearson certification requires iid joint draws,
    so certification is intentionally restricted to batch_size=1.
    """
    if int(batch_size) != 1:
        raise ValueError(
            "Formal joint-randomness certification requires batch_size=1 in "
            "the compact batch-shared-kernel implementation."
        )

    rng = np.random.default_rng(seed)
    all_scores = []
    all_predictions = []
    remaining = int(num_samples)

    while remaining > 0:
        current = min(int(batch_size), remaining)
        noisy = []
        for x in x_modalities:
            repeated = _repeat_one_sample(x, current)
            gaussian = rng.normal(0.0, sigma, size=repeated.shape).astype(np.float32)
            noisy.append(repeated + gaussian)

        internal_seeds = rng.integers(
            0, np.iinfo(np.int32).max, size=(current, 2), dtype=np.int32
        )
        logits = model(
            [tf.convert_to_tensor(v) for v in noisy],
            training=False,
            stochastic_seed=tf.convert_to_tensor(internal_seeds),
        ).numpy()
        all_scores.append(logits)
        all_predictions.append(np.argmax(logits, axis=1))
        remaining -= current

    return np.concatenate(all_scores, axis=0), np.concatenate(all_predictions, axis=0)


def certify_multimodal_sample(
    model: HyWeCSMultimodalResNet,
    x_modalities: Sequence[np.ndarray],
    sigma: float,
    n0: int = 100,
    n: int = 100_000,
    alpha: float = 0.001,
    batch_size: int = 64,
    seed: int = 0,
) -> CertificationResult:
    """Paper-aligned expected-logit estimate and finite-sample RS certificate."""

    model.assert_certificate_ready()
    _, pilot_predictions = sample_joint_scores(
        model, x_modalities, sigma, n0, batch_size, seed
    )
    num_classes = model.paper_config.num_classes
    pilot_counts = np.bincount(pilot_predictions, minlength=num_classes)
    candidate = int(np.argmax(pilot_counts))

    scores, predictions = sample_joint_scores(
        model, x_modalities, sigma, n, batch_size, seed + 1_000_003
    )
    certification_counts = np.bincount(predictions, minlength=num_classes)
    expected_scores = scores.mean(axis=0)
    expected_class = int(np.argmax(expected_scores))
    runner_up = np.max(np.delete(expected_scores, expected_class))
    margin = float(expected_scores[expected_class] - runner_up)
    expected_radius = max(0.0, margin) / (
        2.0 * model.paper_config.global_lipschitz_target
    )

    p_lower = clopper_pearson_lower(
        int(certification_counts[candidate]), int(n), float(alpha)
    )
    if p_lower > 0.5:
        smoothed_class = candidate
        rs_radius = float(sigma * scipy_norm.ppf(p_lower))
        abstained = False
    else:
        smoothed_class = ABSTAIN
        rs_radius = 0.0
        abstained = True

    return CertificationResult(
        expected_logit_class=expected_class,
        expected_logit_radius_mc=expected_radius,
        smoothed_class=smoothed_class,
        randomized_smoothing_radius=rs_radius,
        p_lower=p_lower,
        pilot_counts=pilot_counts,
        certification_counts=certification_counts,
        expected_scores=expected_scores,
        abstained=abstained,
    )


def certified_accuracy_at_radius(
    results: Sequence[CertificationResult],
    labels: Sequence[int],
    radius: float,
) -> float:
    labels = np.asarray(labels, dtype=np.int64)
    if len(results) != len(labels):
        raise ValueError("results and labels must have equal length.")
    correct = [
        (not result.abstained)
        and result.smoothed_class == int(label)
        and result.randomized_smoothing_radius >= float(radius)
        for result, label in zip(results, labels)
    ]
    return 100.0 * float(np.mean(correct))


# =============================================================================
# Audits that prevent silent regression to the legacy design
# =============================================================================


def iter_nested_layers(root: layers.Layer) -> Iterable[layers.Layer]:
    seen: set[int] = set()

    def visit(layer: layers.Layer) -> Iterable[layers.Layer]:
        if id(layer) in seen:
            return
        seen.add(id(layer))
        yield layer
        for child in getattr(layer, "layers", []) or []:
            if isinstance(child, layers.Layer):
                yield from visit(child)
        for value in vars(layer).values():
            if isinstance(value, layers.Layer):
                yield from visit(value)
            elif isinstance(value, (list, tuple)):
                for item in value:
                    if isinstance(item, layers.Layer):
                        yield from visit(item)

    yield from visit(root)


def audit_hywecs_model(model: HyWeCSMultimodalResNet) -> dict[str, Any]:
    """Verify the paper's exact r/(n-r) two-partition parameterization."""

    hywecs_layers = [
        layer for layer in iter_nested_layers(model) if isinstance(layer, HyWeCSConv2D)
    ]
    allocation_checks = []
    parameter_checks = []
    no_duplicate_kernel_checks = []

    for layer in hywecs_layers:
        if (
            layer._oraw_indices_np is None
            or layer._shanw_indices_np is None
            or layer.n_weights_per_filter is None
        ):
            allocation_checks.append(False)
            parameter_checks.append(False)
            no_duplicate_kernel_checks.append(False)
            continue

        oraw = np.asarray(layer._oraw_indices_np, dtype=np.int64)
        shanw = np.asarray(layer._shanw_indices_np, dtype=np.int64)
        cout = int(layer.config.filters)
        n = int(layer.n_weights_per_filter)
        expected_r = int(np.rint(layer.config.oraw_fraction * n))
        expected_r = min(max(expected_r, 1), n - 1)

        per_filter_ok = True
        for out_index in range(cout):
            o = oraw[out_index]
            s = shanw[out_index]
            per_filter_ok = per_filter_ok and (
                len(o) == expected_r
                and len(s) == n - expected_r
                and np.intersect1d(o, s).size == 0
                and np.array_equal(
                    np.sort(np.concatenate([o, s])),
                    np.arange(n, dtype=np.int64),
                )
            )
        allocation_checks.append(bool(per_filter_ok))

        partition_parameter_count = int(np.prod(layer.oraw_base_values.shape)) + int(
            np.prod(layer.shanw_base_values.shape)
        )
        parameter_checks.append(partition_parameter_count == n * cout)
        no_duplicate_kernel_checks.append(not hasattr(layer, "base_kernel"))

    return {
        "hywecs_layer_count": len(hywecs_layers),
        "fixed_within_filter_allocation": all(allocation_checks),
        "exactly_two_weight_partitions": all(allocation_checks),
        "partition_weights_equal_one_standard_kernel": all(parameter_checks),
        "no_duplicate_full_kernel_parameters": all(no_duplicate_kernel_checks),
        "all_layers_verified": all(layer.certificate_ready for layer in hywecs_layers),
        "global_calibration_verified": model.global_calibration.certificate_ready,
        "cross_fusion_verified": model.paper_config.reciprocal_cross_fusion_verified,
        "certificate_ready": model.certificate_ready,
        "uses_separate_feature_noise_module": False,
        "global_lipschitz_target": model.paper_config.global_lipschitz_target,
    }

__all__ = [
    "ABSTAIN",
    "CertificationResult",
    "GlobalScoreCalibration",
    "HybridSpaceWeightAttention",
    "HyWeCSConv2D",
    "HyWeCSLayerConfig",
    "HyWeCSModelConfig",
    "HyWeCSMultimodalResNet",
    "HyWeCSResidualBlock",
    "adversarial_train_step",
    "audit_hywecs_model",
    "build_hywecs_multimodal_resnet18",
    "certification_compatible_train_step",
    "certified_accuracy_at_radius",
    "certify_multimodal_sample",
    "clopper_pearson_lower",
    "eot_pgd_multimodal",
    "load_numpy_multimodal_split",
    "lot_retract_kernel",
    "make_tf_dataset",
    "reshape_vector_modality",
    "sample_joint_scores",
]


In [ ]:
# Runtime and reproducibility audit
SEED = 0
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.keras.backend.clear_session()

print("TensorFlow:", tf.__version__)
print("Physical GPUs:", tf.config.list_physical_devices("GPU"))
print("Mixed precision policy:", tf.keras.mixed_precision.global_policy())
print("The original dataset-loading code above remains unchanged.")
print("Keep float32 for LOT/Newton and hyperbolic numerical stability.")


## 2. Validate the originally loaded dataset

This cell only checks the arrays loaded by the unchanged dataset cell. It performs no file discovery, loading, reshaping, normalization, or replacement.


In [ ]:
# Dataset integrity checks only — no dataset transformation is applied.
_required_arrays = {
    "X_train_image": X_train_image,
    "X_train_ehr": X_train_ehr,
    "y_train": y_train,
    "X_val_image": X_val_image,
    "X_val_ehr": X_val_ehr,
    "y_val": y_val,
    "X_test_image": X_test_image,
    "X_test_ehr": X_test_ehr,
    "y_test": y_test,
}

for _name, _array in _required_arrays.items():
    if not isinstance(_array, np.ndarray):
        raise TypeError(f"{_name} must be a NumPy array; received {type(_array)!r}.")

for _split, _x1, _x2, _y in (
    ("train", X_train_image, X_train_ehr, y_train),
    ("validation", X_val_image, X_val_ehr, y_val),
    ("test", X_test_image, X_test_ehr, y_test),
):
    if not (len(_x1) == len(_x2) == len(_y)):
        raise ValueError(
            f"Sample-count mismatch in {_split}: "
            f"{len(_x1)}, {len(_x2)}, {len(_y)}."
        )

# The provided model is convolutional for both modalities, matching the original notebook.
if X_train_image.ndim != 4 or X_train_ehr.ndim != 4:
    raise ValueError(
        "The original two-branch Conv2D model requires both X_train1.npy and "
        "X_train2.npy to be rank-4 [N,H,W,C] tensors. No automatic reshape is "
        "performed because the original dataset code must remain unchanged."
    )

print("train:", X_train_image.shape, X_train_ehr.shape, y_train.shape)
print("val:  ", X_val_image.shape, X_val_ehr.shape, y_val.shape)
print("test: ", X_test_image.shape, X_test_ehr.shape, y_test.shape)



## 3. Paper-controlled and verifier-controlled configuration

The architecture fixes the **form** of ORaW, SHANW, reciprocal fusion, and `L_star=2`. Numerical choices not uniquely specified in the attached manuscript remain explicit below and must be synchronized with the final experiment configuration.


In [ ]:

def infer_num_classes(labels: np.ndarray) -> int:
    labels = np.asarray(labels)
    if labels.ndim == 2 and labels.shape[-1] > 1:
        return int(labels.shape[-1])
    return int(np.max(labels.reshape(-1))) + 1


NUM_CLASSES = infer_num_classes(y_train)

# Paper/experiment settings: replace with the exact final values used in the manuscript.
ORAW_FRACTION = 0.50
SIGMA_RP = 0.05
SHANW_RECURSIONS = 3
HYPERBOLIC_CURVATURE = 1.0
HYPERBOLIC_DIM = 5  # compact profile; keep identical in all final runs
CHANNEL_BOTTLENECK_RATIO = 8
LOT_NEWTON_STEPS = 4  # finite LOT iteration count; keep identical in all final runs
LOT_EPS = 1e-4
LAYER_TAU = 1.0

# Development mode. It runs the architecture but does not authorize formal certification.
CALIBRATION_MODE = "diagnostic"
FEATURE_NORM_BOUND = None
ATTENTION_LIPSCHITZ_BOUND = None
VERIFIED_NETWORK_BOUND = None
RECIPROCAL_CROSS_FUSION_VERIFIED = False

# Formal mode requires sound externally validated values:
# CALIBRATION_MODE = "verified"
# FEATURE_NORM_BOUND = ...
# ATTENTION_LIPSCHITZ_BOUND = ...
# VERIFIED_NETWORK_BOUND = ...
# RECIPROCAL_CROSS_FUSION_VERIFIED = True

config = HyWeCSModelConfig(
    input_shape_modality_1=tuple(X_train_image.shape[1:]),
    input_shape_modality_2=tuple(X_train_ehr.shape[1:]),
    num_classes=NUM_CLASSES,
    base_width=64,
    blocks_per_stage=(2, 2, 2, 2),
    stage_width_multipliers=(1, 2, 4, 8),
    oraw_fraction=ORAW_FRACTION,
    sigma_rp=SIGMA_RP,
    shanw_recursions=SHANW_RECURSIONS,
    curvature=HYPERBOLIC_CURVATURE,
    hyperbolic_dim=HYPERBOLIC_DIM,
    channel_bottleneck_ratio=CHANNEL_BOTTLENECK_RATIO,
    tau=LAYER_TAU,
    lot_newton_steps=LOT_NEWTON_STEPS,
    lot_eps=LOT_EPS,
    calibration_mode=CALIBRATION_MODE,
    feature_norm_bound=FEATURE_NORM_BOUND,
    attention_lipschitz_bound=ATTENTION_LIPSCHITZ_BOUND,
    global_lipschitz_target=2.0,
    verified_network_bound=VERIFIED_NETWORK_BOUND,
    reciprocal_cross_fusion_verified=RECIPROCAL_CROSS_FUSION_VERIFIED,
    allocation_seed=SEED,
    use_hywecs_stem=False,
    seed=SEED,
)
config.validate()
print(config)


### LOT complex-dtype regression check

This check verifies the Fourier-domain `complex64` Newton--Schulz path before the full model is built.


In [ ]:
# Regression checks for all previously reported TensorFlow failures.

# (1) Fourier-domain complex Newton--Schulz path.
_lot_test_kernel = tf.random.stateless_normal(
    shape=(3, 3, 3, 4),
    seed=tf.constant([SEED, SEED + 97], dtype=tf.int32),
    dtype=tf.float32,
)
_lot_test_output = lot_retract_kernel(
    _lot_test_kernel,
    newton_steps=LOT_NEWTON_STEPS,
    eps=LOT_EPS,
)
tf.debugging.assert_all_finite(
    _lot_test_output,
    "LOT complex-domain regression check produced NaN or Inf.",
)
print(
    "LOT complex-domain regression check passed:",
    _lot_test_output.shape,
    _lot_test_output.dtype,
)

# (2) Exact adaptive-average-pooling forward/backward path.  The graph must
# contain no area-resize operation, and the input gradient must be finite.
_pool_probe = tf.Variable(
    tf.random.stateless_normal(
        [2, 11, 13, 3],
        seed=tf.constant([SEED + 201, SEED + 202], tf.int32),
        dtype=tf.float32,
    )
)
with tf.GradientTape() as _pool_tape:
    _pool_output = adaptive_average_pool2d(_pool_probe, (3, 3))
    _pool_loss = tf.reduce_sum(tf.square(_pool_output))
_pool_gradient = _pool_tape.gradient(_pool_loss, _pool_probe)
if _pool_gradient is None:
    raise RuntimeError("Adaptive average pooling returned no input gradient.")
tf.debugging.assert_all_finite(
    _pool_gradient,
    "Adaptive average-pooling gradient produced NaN or Inf.",
)

@tf.function(autograph=False)
def _adaptive_pool_graph(x):
    return adaptive_average_pool2d(x, (3, 3))

_pool_concrete = _adaptive_pool_graph.get_concrete_function(
    tf.TensorSpec([None, 11, 13, 3], tf.float32)
)
_pool_op_types = {op.type for op in _pool_concrete.graph.get_operations()}
if "ResizeArea" in _pool_op_types:
    raise RuntimeError("A ResizeArea operation remains in the pooling graph.")
print(
    "Adaptive average-pooling gradient check passed:",
    _pool_output.shape,
    "gradient norm=",
    float(tf.linalg.global_norm([_pool_gradient]).numpy()),
)

# (3) Keras 3 graph-scope safety of fixed ORaW/SHANW index partitions.
_probe_cfg = HyWeCSLayerConfig(
    filters=4,
    kernel_size=3,
    strides=1,
    oraw_fraction=ORAW_FRACTION,
    allocation_seed=SEED + 701,
    sigma_rp=SIGMA_RP,
    lot_newton_steps=max(2, min(LOT_NEWTON_STEPS, 4)),
    lot_eps=LOT_EPS,
    shanw_recursions=1,
    curvature=HYPERBOLIC_CURVATURE,
    hyperbolic_dim=min(HYPERBOLIC_DIM, 4),
    channel_bottleneck_ratio=CHANNEL_BOTTLENECK_RATIO,
    tau=LAYER_TAU,
    calibration_mode="diagnostic",
)
_probe_layer = HyWeCSConv2D(_probe_cfg, name="keras3_graph_scope_probe")
_probe_layer.build(tf.TensorShape([None, 8, 8, 3]))

@tf.function(autograph=False)
def _graph_scope_probe(x, seed):
    return _probe_layer(x, training=False, stochastic_seed=seed)

_probe_x = tf.zeros([1, 8, 8, 3], tf.float32)
_probe_seed = tf.constant([SEED + 11, SEED + 12], tf.int32)
_probe_y_1 = _graph_scope_probe(_probe_x, _probe_seed)
_probe_y_2 = _graph_scope_probe(_probe_x, _probe_seed)
tf.debugging.assert_all_finite(
    _probe_y_1, "Keras graph-scope regression probe produced NaN or Inf."
)

# A real backward pass through the complete probe layer verifies gradients
# through LOT, Euclidean/hyperbolic attention, exact adaptive pooling, the
# two-partition assembly, calibration, and convolution.
with tf.GradientTape() as _probe_tape:
    _probe_y_train = _probe_layer(
        _probe_x,
        training=True,
        stochastic_seed=_probe_seed,
    )
    _probe_loss = tf.reduce_mean(tf.square(_probe_y_train))
_probe_gradients = _probe_tape.gradient(
    _probe_loss,
    _probe_layer.trainable_variables,
)
_missing_probe_gradients = [
    variable.name
    for gradient, variable in zip(_probe_gradients, _probe_layer.trainable_variables)
    if gradient is None
]
if _missing_probe_gradients:
    raise RuntimeError(
        "HyWeCS probe has missing gradients for: "
        + ", ".join(_missing_probe_gradients)
    )
for _gradient in _probe_gradients:
    tf.debugging.assert_all_finite(
        tf.convert_to_tensor(_gradient),
        "HyWeCS probe gradient produced NaN or Inf.",
    )
tf.debugging.assert_equal(tf.shape(_probe_y_1), tf.shape(_probe_y_2))
assert isinstance(_probe_layer._oraw_indices_np, np.ndarray)
assert isinstance(_probe_layer._shanw_indices_np, np.ndarray)
assert (_probe_layer.r_weights_per_filter +
        _probe_layer.shanw_weights_per_filter ==
        _probe_layer.n_weights_per_filter)
assert int(np.prod(_probe_layer.oraw_base_values.shape) +
           np.prod(_probe_layer.shanw_base_values.shape)) == (
           _probe_layer.n_weights_per_filter * _probe_cfg.filters)
print("Keras 3 graph-scope and exact two-partition check passed:",
      _probe_y_1.shape)

# (4) Complete HyWeCS layer backward pass, including LOT, Euclidean/hyperbolic
# attention, two-partition assembly, calibration, and convolution.
_probe_x_grad = tf.Variable(
    tf.random.stateless_normal(
        [2, 8, 8, 3],
        seed=tf.constant([SEED + 301, SEED + 302], tf.int32),
        dtype=tf.float32,
    )
)
with tf.GradientTape() as _hywecs_tape:
    _probe_y_grad = _probe_layer(
        _probe_x_grad,
        training=True,
        stochastic_seed=tf.constant([SEED + 41, SEED + 42], tf.int32),
    )
    _probe_loss = tf.reduce_mean(tf.square(_probe_y_grad))
_probe_sources = [_probe_x_grad] + list(_probe_layer.trainable_variables)
_probe_gradients = _hywecs_tape.gradient(_probe_loss, _probe_sources)
if _probe_gradients[0] is None:
    raise RuntimeError("The complete HyWeCS layer returned no input gradient.")
_finite_probe_gradients = [g for g in _probe_gradients if g is not None]
if not _finite_probe_gradients:
    raise RuntimeError("The complete HyWeCS layer returned no gradients.")
for _gradient in _finite_probe_gradients:
    tf.debugging.assert_all_finite(
        _gradient,
        "The complete HyWeCS backward pass produced NaN or Inf.",
    )
print(
    "Complete HyWeCS backward-pass check passed:",
    len(_finite_probe_gradients),
    "finite gradients; global norm=",
    float(tf.linalg.global_norm(_finite_probe_gradients).numpy()),
)

del _lot_test_kernel, _lot_test_output
del _pool_probe, _pool_output, _pool_loss, _pool_gradient, _pool_concrete, _pool_op_types
del _probe_x, _probe_seed, _probe_y_1, _probe_y_2, _probe_x_grad, _probe_y_grad
del _probe_loss, _probe_sources, _probe_gradients, _finite_probe_gradients
del _probe_layer, _probe_cfg


## 4. Build, audit, and profile the compact HyWeCS multimodal model

The parameter count uses the actual Keras variables. The reported GFLOPs follow the standard model-complexity convention and exclude the LOT reparameterization itself; full stochastic inference latency includes ORaW LOT and SHANW realization.


In [ ]:
model = build_hywecs_multimodal_resnet18(config)
model.summary()

_model_probe_1 = tf.zeros([1, *config.input_shape_modality_1], tf.float32)
_model_probe_2 = tf.zeros([1, *config.input_shape_modality_2], tf.float32)
_model_probe_seed = tf.constant([SEED + 101, SEED + 102], tf.int32)

model.set_profile_data_path(True)
try:
    @tf.function(autograph=False)
    def _compiled_model_probe(x1, x2, seed):
        return model([x1, x2], training=False, stochastic_seed=seed)

    _model_probe_logits = _compiled_model_probe(
        _model_probe_1, _model_probe_2, _model_probe_seed
    )
    tf.debugging.assert_all_finite(
        _model_probe_logits, "Compiled HyWeCS model produced NaN or Inf."
    )
finally:
    model.set_profile_data_path(False)
print("End-to-end compact data-path tf.function check passed.")


def _parameter_count(weights):
    return int(sum(np.prod(tuple(int(d) for d in w.shape)) for w in weights))


def profile_tensorflow_data_path_flops(model, input_shapes, seed=0):
    """Profile batch-1 predictor FLOPs without counting LOT reparameterization.

    Model-complexity tables conventionally count the forward data path. LOT is
    a stochastic weight-realization/reparameterization operation; including its
    Newton iterations in the graph produced the misleading 511G number. The
    full stochastic runtime is benchmarked separately below in milliseconds.
    SHANW attention/noise and all convolutions remain active in this profile.
    """
    input_specs = [
        tf.TensorSpec((1, *tuple(shape)), tf.float32, name=f"input_{i+1}")
        for i, shape in enumerate(input_shapes)
    ]
    fixed_seed = tf.constant([int(seed), int(seed) + 1], tf.int32)
    model.set_profile_data_path(True)

    @tf.function(autograph=False)
    def forward_pass(modality_1, modality_2):
        return model(
            [modality_1, modality_2], training=False,
            stochastic_seed=fixed_seed,
        )

    try:
        concrete = forward_pass.get_concrete_function(*input_specs)
        options = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
        options["output"] = "none"
        try:
            from tensorflow.python.framework.convert_to_constants import (
                convert_variables_to_constants_v2_as_graph,
            )
            _, graph_def = convert_variables_to_constants_v2_as_graph(concrete)
            with tf.Graph().as_default() as frozen_graph:
                tf.graph_util.import_graph_def(graph_def, name="")
                profile = tf.compat.v1.profiler.profile(
                    graph=frozen_graph, cmd="op", options=options
                )
            backend = "frozen TensorFlow graph"
        except Exception as freeze_error:
            print("Frozen profiler unavailable; using concrete graph:", freeze_error)
            profile = tf.compat.v1.profiler.profile(
                graph=concrete.graph, cmd="op", options=options
            )
            backend = "concrete TensorFlow graph"
        if profile is None or profile.total_float_ops is None:
            raise RuntimeError("TensorFlow profiler did not return a FLOP count.")
        return int(profile.total_float_ops), backend
    finally:
        model.set_profile_data_path(False)


def benchmark_inference_ms(
    model, x1, x2, include_lot=False, warmup=2, repeats=10
):
    """GPU-synchronized batch-1 latency.

    ``include_lot=False`` matches the forward-path complexity convention.
    Set ``include_lot=True`` to additionally time stochastic ORaW LOT
    realization; this is intentionally optional because exact LOT is costly.
    """
    import time
    model.set_profile_data_path(not bool(include_lot))
    try:
        seed = tf.constant([SEED + 301, SEED + 302], tf.int32)
        for _ in range(int(warmup)):
            y = model([x1, x2], training=False, stochastic_seed=seed)
            _ = y.numpy()
        timings = []
        for repeat in range(int(repeats)):
            s = tf.constant([SEED + 401 + repeat, SEED + 901 + repeat], tf.int32)
            t0 = time.perf_counter()
            y = model([x1, x2], training=False, stochastic_seed=s)
            _ = y.numpy()
            timings.append((time.perf_counter() - t0) * 1000.0)
        return float(np.mean(timings)), float(np.std(timings))
    finally:
        model.set_profile_data_path(False)


total_params = int(model.count_params())
trainable_params = _parameter_count(model.trainable_weights)
non_trainable_params = _parameter_count(model.non_trainable_weights)
flops, flops_backend = profile_tensorflow_data_path_flops(
    model,
    (config.input_shape_modality_1, config.input_shape_modality_2),
    seed=SEED,
)
latency_mean, latency_std = benchmark_inference_ms(
    model, _model_probe_1, _model_probe_2,
    include_lot=False, warmup=3, repeats=20,
)
RUN_FULL_LOT_LATENCY_BENCHMARK = False
if RUN_FULL_LOT_LATENCY_BENCHMARK:
    lot_latency_mean, lot_latency_std = benchmark_inference_ms(
        model, _model_probe_1, _model_probe_2,
        include_lot=True, warmup=1, repeats=3,
    )
else:
    lot_latency_mean = lot_latency_std = None

print("\n" + "=" * 76)
print("HYWECS COMPLEXITY — COMPUTED BEFORE TRAINING")
print("=" * 76)
print(f"Total parameters:         {total_params:,} ({total_params / 1e6:.4f} M)")
print(f"Trainable parameters:     {trainable_params:,} ({trainable_params / 1e6:.4f} M)")
print(f"Non-trainable parameters: {non_trainable_params:,} ({non_trainable_params / 1e6:.4f} M)")
print(f"Forward-path FLOPs:       {flops:,} ({flops / 1e9:.4f} G)")
print(f"Forward-path latency:     {latency_mean:.3f} ± {latency_std:.3f} ms")
if lot_latency_mean is not None:
    print(f"Full LOT latency:         {lot_latency_mean:.3f} ± {lot_latency_std:.3f} ms")
else:
    print("Full LOT latency:         not run (set RUN_FULL_LOT_LATENCY_BENCHMARK=True)")
print(f"FLOP profiler backend:    {flops_backend}")
print("FLOPs cover one complete two-modality batch-1 forward pass and")
print("exclude LOT weight reparameterization; optional full LOT latency includes it.")
print("=" * 76)

# Guard against any duplicated full-kernel parameterization.
if total_params > 40_000_000:
    raise RuntimeError(
        "Parameter budget regression: the two ORaW/SHANW partitions must not "
        "duplicate full convolution kernels."
    )

audit = audit_hywecs_model(model)
for key, value in audit.items():
    print(f"{key}: {value}")
assert audit["fixed_within_filter_allocation"]
assert audit["exactly_two_weight_partitions"]
assert audit["partition_weights_equal_one_standard_kernel"]
assert audit["no_duplicate_full_kernel_parameters"]
assert audit["uses_separate_feature_noise_module"] is False
assert audit["global_lipschitz_target"] == 2.0
print("Structural HyWeCS audit passed.")


## 5. Data pipeline and optimizer

In [ ]:

DOMAIN = "natural"  # choose "natural" or "medical"
EPOCHS = 200
BATCH_SIZE = 64      # paper setting; reduce only if the dataset/runtime requires it
SIGMA = 0.25
Q_TRAIN = 1

train_ds = make_tf_dataset(
    X_train_image, X_train_ehr, y_train,
    batch_size=BATCH_SIZE, shuffle=True, seed=SEED,
)
val_ds = make_tf_dataset(
    X_val_image, X_val_ehr, y_val,
    batch_size=BATCH_SIZE, shuffle=False,
)
test_ds = make_tf_dataset(
    X_test_image, X_test_ehr, y_test,
    batch_size=BATCH_SIZE, shuffle=False,
)

if DOMAIN == "natural":
    BASE_LR = 1e-2
    LR_STEP_SIZE = 30
    LR_GAMMA = 0.1
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=BASE_LR,
        weight_decay=1e-5,
    )
elif DOMAIN == "medical":
    BASE_LR = 5e-2
    LR_STEP_SIZE = 30
    LR_GAMMA = 0.8
    optimizer = tf.keras.optimizers.SGD(learning_rate=BASE_LR)
else:
    raise ValueError("DOMAIN must be 'natural' or 'medical'.")

print("Optimizer:", optimizer.__class__.__name__)


In [ ]:

def set_step_learning_rate(optimizer, epoch, base_lr, step_size, gamma):
    lr = float(base_lr) * (float(gamma) ** (int(epoch) // int(step_size)))
    if hasattr(optimizer.learning_rate, "assign"):
        optimizer.learning_rate.assign(lr)
    else:
        optimizer.learning_rate = lr
    return lr


def labels_to_indices(labels):
    labels = tf.convert_to_tensor(labels)
    if labels.shape.rank == 2 and labels.shape[-1] > 1:
        return tf.argmax(labels, axis=-1, output_type=tf.int64)
    return tf.cast(tf.reshape(labels, [-1]), tf.int64)


def stochastic_mean_logits(model, modalities, mc_samples=1):
    batch_size = tf.shape(modalities[0])[0]
    total = None
    for _ in range(int(mc_samples)):
        seeds = random_seed_batch(batch_size)
        logits = model(modalities, training=False, stochastic_seed=seeds)
        total = logits if total is None else total + logits
    return total / float(mc_samples)


def evaluate_accuracy(model, dataset, mc_samples=1):
    correct = 0
    total = 0
    for modalities, labels in dataset:
        modalities = [tf.cast(x, tf.float32) for x in modalities]
        logits = stochastic_mean_logits(model, modalities, mc_samples=mc_samples)
        pred = tf.argmax(logits, axis=-1, output_type=tf.int64)
        target = labels_to_indices(labels)
        correct += int(tf.reduce_sum(tf.cast(pred == target, tf.int32)).numpy())
        total += int(tf.size(target).numpy())
    return correct / max(total, 1)



## 6. One-batch smoke test

Run this before a long Kaggle job. It verifies the complete forward/backward path with Gaussian input smoothing and joint ORaW–SHANW randomness.


In [ ]:

(inputs_batch, labels_batch) = next(iter(train_ds))
smoke_metrics = certification_compatible_train_step(
    model=model,
    optimizer=optimizer,
    modalities=list(inputs_batch),
    labels=labels_batch,
    sigma=SIGMA,
    q_tr=Q_TRAIN,
)
print({key: float(value.numpy()) for key, value in smoke_metrics.items()})


## 7. Full training loop with best-checkpoint saving

In [ ]:

RUN_FULL_TRAINING = False
USE_ADVERSARIAL_TRAINING = False

ADV_EPSILON = 8.0 / 255.0
ADV_STEP_SIZE = 2.0 / 255.0
ADV_STEPS = 20
Q_ATTACK_TRAIN = 1  # one fixed internal realization per PGD training trajectory
Q_UPDATE = 1
LAMBDA_ADV = 1.0
CLIP_VALUES = [None, None]  # Set modality-specific (min,max) only when justified.

CHECKPOINT_PATH = "/kaggle/working/hywecs_best.weights.h5"
MC_VALIDATION_SAMPLES = 1

history = []
best_val_accuracy = -np.inf

if RUN_FULL_TRAINING:
    import time
    for epoch in range(EPOCHS):
        epoch_t0 = time.perf_counter()
        lr = set_step_learning_rate(
            optimizer, epoch, BASE_LR, LR_STEP_SIZE, LR_GAMMA
        )
        running = []

        for modalities, labels in train_ds:
            if USE_ADVERSARIAL_TRAINING:
                metrics = adversarial_train_step(
                    model=model,
                    optimizer=optimizer,
                    modalities=list(modalities),
                    labels=labels,
                    sigma=SIGMA,
                    epsilon=ADV_EPSILON,
                    step_size=ADV_STEP_SIZE,
                    attack_steps=ADV_STEPS,
                    q_attack=Q_ATTACK_TRAIN,
                    q_update=Q_UPDATE,
                    lambda_adv=LAMBDA_ADV,
                    clip_values=CLIP_VALUES,
                )
            else:
                metrics = certification_compatible_train_step(
                    model=model,
                    optimizer=optimizer,
                    modalities=list(modalities),
                    labels=labels,
                    sigma=SIGMA,
                    q_tr=Q_TRAIN,
                )
            running.append(float(metrics["loss"].numpy()))

        val_accuracy = evaluate_accuracy(
            model, val_ds, mc_samples=MC_VALIDATION_SAMPLES
        )
        epoch_record = {
            "epoch": epoch + 1,
            "learning_rate": lr,
            "train_loss": float(np.mean(running)),
            "val_accuracy": val_accuracy,
            "epoch_time_s": float(time.perf_counter() - epoch_t0),
            "training_mode": "adversarial" if USE_ADVERSARIAL_TRAINING else "certification-compatible",
        }
        history.append(epoch_record)
        print(epoch_record)

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            model.save_weights(CHECKPOINT_PATH)
            print("Saved best checkpoint:", CHECKPOINT_PATH)
else:
    print("Set RUN_FULL_TRAINING=True to launch the 200-epoch Kaggle run.")


## 8. Load the best checkpoint and evaluate clean accuracy

In [ ]:

from pathlib import Path

if Path(CHECKPOINT_PATH).exists():
    model.load_weights(CHECKPOINT_PATH)
    print("Loaded:", CHECKPOINT_PATH)
else:
    print("No saved checkpoint yet; evaluating the current in-memory model.")

MC_TEST_SAMPLES = 5
test_accuracy = evaluate_accuracy(model, test_ds, mc_samples=MC_TEST_SAMPLES)
print(f"Clean stochastic-mean test accuracy: {100.0 * test_accuracy:.2f}%")



## 9. Optional EOT-PGD empirical robustness evaluation

The attack keeps a set of sampled Gaussian/internal realizations fixed through one PGD trajectory, matching the paper’s attack-phase protocol. Use the exact clipping range of each modality when preprocessing defines one.


In [ ]:

def evaluate_eot_pgd_accuracy(
    model,
    dataset,
    sigma,
    epsilon,
    step_size,
    steps,
    q_attack,
    clip_values=None,
    max_batches=None,
):
    correct = 0
    total = 0
    for batch_index, (modalities, labels) in enumerate(dataset):
        if max_batches is not None and batch_index >= int(max_batches):
            break
        clean = [tf.cast(x, tf.float32) for x in modalities]
        deltas = eot_pgd_multimodal(
            model=model,
            modalities=clean,
            labels=labels,
            sigma=sigma,
            epsilon=epsilon,
            step_size=step_size,
            steps=steps,
            q_attack=q_attack,
            clip_values=clip_values,
        )
        attacked = [x + delta for x, delta in zip(clean, deltas)]
        logits = stochastic_mean_logits(model, attacked, mc_samples=q_attack)
        pred = tf.argmax(logits, axis=-1, output_type=tf.int64)
        target = labels_to_indices(labels)
        correct += int(tf.reduce_sum(tf.cast(pred == target, tf.int32)).numpy())
        total += int(tf.size(target).numpy())
    return correct / max(total, 1)


Q_ATTACK_EVAL = 10  # paper evaluation setting
RUN_EOT_PGD_EVALUATION = False
if RUN_EOT_PGD_EVALUATION:
    robust_accuracy = evaluate_eot_pgd_accuracy(
        model=model,
        dataset=test_ds,
        sigma=SIGMA,
        epsilon=ADV_EPSILON,
        step_size=ADV_STEP_SIZE,
        steps=ADV_STEPS,
        q_attack=Q_ATTACK_EVAL,
        clip_values=CLIP_VALUES,
    )
    print(f"EOT-PGD robust accuracy: {100.0 * robust_accuracy:.2f}%")
else:
    print("Set RUN_EOT_PGD_EVALUATION=True after loading the trained checkpoint.")



## 10. Inference-time certification under full joint randomness

Certification jointly resamples Gaussian input noise, ORaW random-projection weights, and Euclidean/hyperbolic SHANW noise. It intentionally refuses to run in diagnostic mode. Switch to verified mode only after supplying sound verifier outputs.


In [ ]:

N0 = 100
N_CERTIFY = 100_000
ALPHA = 0.001
CERTIFICATION_BATCH_SIZE = 1

print("certificate_ready:", model.certificate_ready)

RUN_FORMAL_CERTIFICATION = False
if RUN_FORMAL_CERTIFICATION:
    result = certify_multimodal_sample(
        model=model,
        x_modalities=[X_test_image[0], X_test_ehr[0]],
        sigma=SIGMA,
        n0=N0,
        n=N_CERTIFY,
        alpha=ALPHA,
        batch_size=CERTIFICATION_BATCH_SIZE,
        seed=12345,
    )
    print(result)
else:
    print(
        "Set verified bounds, rebuild the model with calibration_mode='verified', "
        "then set RUN_FORMAL_CERTIFICATION=True."
    )
